# 01. 알약 데이터 EDA (통합본)

팀원별 EDA 노트북 6개를 주제별로 합친 노트북입니다. 같은 주제는 한 셀을 기준본으로 두고
나머지 노트북의 장점을 기준본에 녹였습니다. 섹션마다 첫 셀에 원작자와 원본 노트북을 적었습니다.

## 실행 방법

- `uv sync` 후 VS Code나 `uv run jupyter lab`에서 프로젝트 `.venv` 커널로 엽니다.
- 위에서 아래로 한 번 실행(Restart & Run All)합니다. 작업 폴더는 프로젝트 루트와 `notebooks/` 모두 됩니다.
- 원본은 `data/raw/`에서 읽기만 합니다. `ann_df`·`img_df`·`cat_df`는 원본 상태로 유지하고,
  보정 비교는 별도 `ann_corrected_df`에서 합니다.
- 커밋 전에 출력을 지웁니다(Clear All Outputs).

## 실행 옵션 (1장 설정 셀)

| 옵션 | 기본값 | 설명 |
|---|---|---|
| `RUN_HEAVY` | `False` | 전체 이미지 SHA256·디코딩, 전체 박스 GrabCut 같은 무거운 연산을 실행합니다. `False`이면 입력·버전·파라미터가 일치하는 캐시만 재사용하고, 없으면 `미실행`으로 표시합니다. `True`이면 캐시와 관계없이 다시 계산합니다. |
| `SAVE_OUTPUTS` | `False` | 그림·보고서·캐시 파일을 저장합니다. `False`이면 파일을 쓰지 않습니다. 켜도 원본은 수정하지 않습니다. |
| `SEED` | `42` | 무작위 연산 전체에 쓰는 seed입니다. |

`미실행` 결과는 0건이나 검사 통과로 해석하지 않습니다. 후속 셀은 결과 상태를 확인한 뒤 집계합니다.

## 데이터 상태 표시

표와 그림 제목 앞에 데이터 상태를 붙입니다. `[원본]`은 `data/raw/`를 그대로 읽은 결과,
`[보정]`은 보정 기록을 메모리에서 적용한 결과입니다.

## 섹션별 실행 시간

| 섹션 | `RUN_HEAVY=False` | `RUN_HEAVY=True` |
|---|---|---|
| 1–2. 설정·데이터 로드 | 약 3초 (Apple Silicon Mac) | 같음 |
| 3–4. 규모·구조·결측 | 약 2초 | 약 15초 (전체 디코딩·SHA256 13초) |
| 5. 라벨 오류와 이상치 | 약 12초 (박스 픽셀 측정 10초) | 약 2분 10초 (GrabCut 여백 추정 2분) |
| 6–9 | 이식 후 측정 | 이식 후 측정 |

## 목차

0. 개요
1. 환경 설정
2. 데이터 로드
3. 데이터 규모와 구조
4. 결측
5. 라벨 오류와 이상치
6. 분포
7. 촬영 조건
8. 데이터 누수와 분할
9. 결론과 다음 작업

## 원본 노트북

| 노트북 | 작성자 |
|---|---|
| `EDA_jang.ipynb` | younghoon1215 |
| `JJAM99_eda.ipynb` | potato |
| `dongwook_eda.ipynb` | def-uk |
| `eda_kwon.ipynb` | gwon753-dev |
| `eda_sohee.ipynb`, `sohee_eda.ipynb` | bravedolphinhere |

## 1. 환경 설정

> 출처: dongwook_eda.ipynb 셀 1 (def-uk), eda_kwon.ipynb 셀 2, 35 (gwon753-dev),
> sohee_eda.ipynb 셀 1, 3 (bravedolphinhere)

- 경로는 모두 `src/config.py`의 `PROJECT_ROOT` 기준입니다.
- 한글 폰트는 Malgun Gothic → AppleGothic → NanumGothic → Noto Sans CJK KR 순서로 찾습니다.
  WSL에서는 Windows의 맑은 고딕 파일을 등록합니다.
- 무작위 연산은 모두 `SEED`를 씁니다(`random_state=SEED`, `np.random.default_rng(SEED)`).

In [ ]:
import sys
from pathlib import Path

# 노트북은 notebooks/에서 실행되므로 프로젝트 루트를 import 경로에 추가한다.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import hashlib
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from data_process.data_processing import (
    BBOX_COLS,
    PADDING_VERSION,
    bbox_flags,
    bbox_iou,
    build_cat_df,
    build_img_df,
    compare_name_codes,
    find_box_pairs,
    fingerprint_files,
    inspect_image_integrity,
    iqr_bounds,
    list_images,
    load_annotations,
    load_cache,
    load_schema,
    measure_bbox_padding,
    measure_box_pixels,
    propose_bbox_at_center,
    propose_bbox_by_shape,
    read_image_headers,
    save_cache,
    scan_missing_values,
)
from data_process.make_labels import (
    CORRECTION_COLS,
    STAGES,
    apply_corrections,
    assign_new_ids,
    correction_for_label,
    corrections_from_kwon,
)
from src.config import PROJECT_ROOT
from visualization import (
    draw_bbox,
    draw_bboxes,
    draw_image_bounds,
    image_grid,
    set_korean_font,
    show_image,
    zoom_to,
)

SEED = 42
RUN_HEAVY = False  # True면 무거운 연산을 캐시와 관계없이 다시 계산한다.
SAVE_OUTPUTS = False  # True일 때만 그림·보고서·캐시 파일을 쓴다.
NOT_RUN = "미실행"  # 생략한 분석의 결과 상태. 0건이나 통과로 해석하지 않는다.

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"  # 받은 원본 그대로 (읽기만 함)
# 압축을 푼 위치에 따라 sprint_ai_project1_data 폴더가 한 단계 더 있을 수 있다.
if (RAW_DIR / "sprint_ai_project1_data").is_dir():
    RAW_DIR = RAW_DIR / "sprint_ai_project1_data"
PROCESSED_DIR = DATA_DIR / "processed"  # 보정본·보정 기록
CACHE_DIR = PROCESSED_DIR / "eda_cache"  # 무거운 연산 캐시. SAVE_OUTPUTS=True일 때만 쓴다.

TRAIN_IMG_DIR = RAW_DIR / "train_images"
TRAIN_ANN_DIR = RAW_DIR / "train_annotations"
TEST_IMG_DIR = RAW_DIR / "test_images"
for folder in [TRAIN_IMG_DIR, TRAIN_ANN_DIR, TEST_IMG_DIR]:
    if not folder.is_dir():
        raise FileNotFoundError(f"데이터 폴더가 없습니다: {folder.relative_to(PROJECT_ROOT)}")

np.random.seed(SEED)
pd.set_option("display.max_columns", 60)
font = set_korean_font()
if font is None:
    print("한글 폰트를 찾지 못했습니다. 약 이름이 네모로 보일 수 있습니다.")

print(f"SEED={SEED}, RUN_HEAVY={RUN_HEAVY}, SAVE_OUTPUTS={SAVE_OUTPUTS}, 폰트={font}")
print(f"원본 데이터: {RAW_DIR.relative_to(PROJECT_ROOT).as_posix()}/")

## 2. 데이터 로드

> 출처: dongwook_eda.ipynb 셀 0–3 (def-uk) 기준. eda_kwon.ipynb 셀 1–2 (gwon753-dev)의
> 디스크 기준 `img_df`·`cat_df`·`test_files`, sohee_eda.ipynb 셀 2, 7 (bravedolphinhere)의
> JSON 내부 참조 연결과 읽기 오류 기록을 합쳤습니다. 로더는 `data_process/data_processing.py`에 있습니다.

- annotation JSON 하나 = 이미지 1장 × 알약 1개입니다. 같은 이미지의 알약이 약 코드별 폴더에 흩어져 있습니다.
  경로: `train_annotations/<조합>_json/<약코드>/<이미지이름>.json`
- bbox는 COCO 형식 `[x, y, w, h]`(왼쪽 위 기준, 픽셀)입니다.
- 이미지 메타데이터는 JSON 안의 `annotation.image_id`로 연결합니다. 전역 `image_id`로 합치지 않습니다.
- 라벨 식별자는 `(source_json, annotation_id)`입니다. `source_json`은 `train_annotations/` 기준 상대 경로입니다.
- bbox 값이 비어 있는 라벨도 빼지 않고 `x, y, w, h`를 NaN으로 둡니다(4.1에서 확인).
- 읽기·연결 실패는 `load_error_df`에 남기고, 분석에서 제외된 입력 수를 보여 줍니다.

| 표 | 한 행 | 설명 |
|---|---|---|
| `ann_df` | 원본 bbox 1개 | 이미지 메타데이터 + 약 정보 + bbox. 원본 상태를 유지합니다. |
| `img_df` | 디스크의 train 이미지 1장 | 라벨 없는 이미지도 포함합니다(`n_boxes=0`). `combo`는 파일 이름의 알약 조합입니다. |
| `cat_df` | 클래스 1개 | `category_id` ↔ 약 이름, bbox 수, 이미지 수 |
| `test_files` | test 이미지 1장 | 파일 이름 목록 |
| `load_error_df` | 오류 1건 | `stage`: `parse`(JSON 읽기), `image_link`, `category_link`, `empty`(annotation 없음) |

In [ ]:
ann_df, load_error_df = load_annotations(TRAIN_ANN_DIR)
img_df = build_img_df(ann_df, list_images(TRAIN_IMG_DIR))
cat_df = build_cat_df(ann_df)
test_files = [p.name for p in list_images(TEST_IMG_DIR)]

# 제외된 입력: parse 오류는 JSON 파일 전체, image_link 오류는 annotation 하나가 빠진다.
n_json = sum(1 for _ in TRAIN_ANN_DIR.rglob("*.json"))
excluded = load_error_df[load_error_df["excluded"]]
n_json_excluded = (excluded["stage"] == "parse").sum()
n_ann_excluded = (excluded["stage"] == "image_link").sum()
n_no_image = (~ann_df["file_name"].isin(img_df["file_name"])).sum()

print(f"[원본] annotation JSON: {n_json}개 (읽기 실패로 제외: {n_json_excluded}개)")
print(
    f"[원본] bbox: {len(ann_df)}개 (연결 실패로 제외: {n_ann_excluded}개), 클래스: {len(cat_df)}개"
)
print(
    f"[원본] train 이미지: {len(img_df)}장 (bbox 없음: {(img_df['n_boxes'] == 0).sum()}장), "
    f"test 이미지: {len(test_files)}장"
)
print(f"[원본] 디스크에 이미지 파일이 없는 bbox: {n_no_image}개")
print(f"로드 오류: {len(load_error_df)}건")
if not load_error_df.empty:
    display(load_error_df)

In [ ]:
display(ann_df.head(3))
display(img_df.head(3))
display(cat_df.head(3))

### 2.1 데이터 정의서

> 출처: eda_kwon.ipynb 셀 3 (gwon753-dev) 기준. dongwook_eda.ipynb 셀 9 (def-uk)의
> 스키마 필수여부 읽기를 합쳤습니다.

먼저 `data/README.md`의 스키마 표와 실제 JSON 키를 대조하고, `ann_df` 컬럼마다 필수여부(M 필수,
O 선택, C 조건부)와 스키마 설명을 붙입니다. 결측 비율은 4.3에서 봅니다.

In [ ]:
schema_df = load_schema(DATA_DIR / "README.md")
# 최상위 키(images, annotations 등)는 field가 비어 있어 section만 키로 쓴다.
schema_df["key"] = (schema_df["section"] + "." + schema_df["field"]).str.rstrip(".")

# ann_df 컬럼 → 스키마 키. 로더가 바꾸거나 나눈 컬럼만 적고, 나머지는 images[]의 같은 이름.
LOADER_COLS = {
    "image_id": "images.id",
    "source_json": None,  # 로더가 붙인 원본 JSON 상대 경로
    "annotation_id": "annotations.id",
    "category_id": "annotations.category_id",
    "category_name": "categories.name",
    "x": "annotations.bbox",
    "y": "annotations.bbox",
    "w": "annotations.bbox",
    "h": "annotations.bbox",
    "area": "annotations.area",
    "iscrowd": "annotations.iscrowd",
    "ignore": "annotations.ignore",
    "has_segmentation": "annotations.segmentation",
}
col_to_key = {c: LOADER_COLS.get(c, f"images.{c}") for c in ann_df.columns}

json_image_keys = {k for c, k in col_to_key.items() if k and k.startswith("images.")}
is_image_field = (schema_df["section"] == "images") & (schema_df["field"] != "")
schema_image_keys = set(schema_df.loc[is_image_field, "key"])
print("스키마에 없는 JSON 키:", sorted(json_image_keys - schema_image_keys) or "없음")
print("JSON에 없는 스키마 키:", sorted(schema_image_keys - json_image_keys) or "없음")

definition_df = pd.DataFrame(
    {
        "컬럼": ann_df.columns,
        "스키마 키": [col_to_key[c] for c in ann_df.columns],
        "dtype": ann_df.dtypes.astype(str).to_numpy(),
        "고유값 수": ann_df.nunique().to_numpy(),
        "예시": [ann_df[c].dropna().iloc[0] if ann_df[c].notna().any() else None for c in ann_df],
    }
)
definition_df = definition_df.merge(
    schema_df[["key", "type", "required", "description"]],
    left_on="스키마 키",
    right_on="key",
    how="left",
).drop(columns="key")
definition_df = definition_df.rename(
    columns={"type": "스키마 타입", "required": "필수여부", "description": "스키마 설명"}
)
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 40):
    display(definition_df)

아래는 kwon이 값을 보고 정리한 컬럼 해석입니다. 수치는 원본 상태 관측값이며 3–4장에서 다시 확인합니다.
필수여부는 위 표를 따릅니다.

### 이미지 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| file_name | str | 이미지 파일명. 이미지 단위의 키로 사용됩니다. 앞부분(`K-003351-018357-036637`)에 사진에 찍힌 약 코드가 들어 있습니다. |
| imgfile | str | 이미지 파일명. `file_name`과 같은 값입니다. 스키마 표에는 없는 키입니다. |
| width | int | 이미지 너비(px). 전부 976입니다. |
| height | int | 이미지 높이(px). 전부 1280입니다. |
| image_id | int | JSON 안의 이미지 아이디(`images[].id`)입니다. |
| source_json | str | 이 행을 읽어 온 annotation JSON의 상대 경로입니다(`train_annotations/` 기준). |

### 촬영 조건

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| back_color | str | 배경색. 전부 `연회색 배경`입니다. |
| light_color | str | 조명 색. 전부 `주백색`입니다. |
| camera_la | int | 카메라 촬영 각도(위도). 70, 75, 90 세 가지입니다. |
| camera_lo | int | 카메라 회전 각도(경도). 전부 0입니다. |
| size | int | 촬영 크기 설정값. 전부 200입니다. |
| drug_dir | str | 촬영된 알약 면. `앞면` 또는 `뒷면`입니다. |
| drug_S | str | 알약 상태. 전부 `정상알약`입니다. |

### 정답 (타깃)

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| annotation_id | int | 어노테이션 아이디. `source_json`과 함께 라벨 하나를 가리킵니다. |
| category_id | int | **타깃 변수(클래스)**. 약 종류 코드이며 56종입니다. `drug_N`의 숫자 부분과 같습니다. |
| category_name | str | JSON `categories[].name`의 약 이름입니다. |
| x | int | **타깃 변수(bbox)**. 박스 왼쪽 위 꼭짓점의 x 좌표(px)입니다. |
| y | int | **타깃 변수(bbox)**. 박스 왼쪽 위 꼭짓점의 y 좌표(px)입니다. |
| w | int | **타깃 변수(bbox)**. 박스 너비(px)입니다. |
| h | int | **타깃 변수(bbox)**. 박스 높이(px)입니다. |
| area | int | 박스 넓이(px²). 대부분 `w × h`와 같습니다. 다른 경우는 5.4에서 봅니다. |
| iscrowd, ignore, has_segmentation | int, int, bool | 평가 분류, 무시 여부, segmentation 유무입니다. |

### 약 식별 코드

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| drug_N | str | 약 코드(예: `K-003351`). `category_id`에 `K-`를 붙인 형태입니다. |
| dl_mapping_code | str | 약 매핑 코드. `drug_N`과 같은 값입니다. |
| dl_idx | str | 약 코드의 숫자 부분(예: `3351`)입니다. (이 부분은 일단 눈여겨 보자) |
| item_seq | int | 식약처 품목기준코드입니다. |
| di_edi_code | str | 건강보험 EDI 코드. 여러 개면 쉼표로 구분합니다. |

### 약 기본 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| dl_name | str | 약 이름(한글)입니다. |
| dl_name_en | str | 약 이름(영문). 14행이 빈 문자열입니다. |
| dl_material | str | 주성분(한글). 성분이 여러 개면 `\|`로 구분합니다. 9행이 빈 문자열입니다. |
| dl_material_en | str | 주성분(영문)입니다. |
| dl_company | str | 제약회사(한글)입니다. |
| dl_company_en | str | 제약회사(영문)입니다. |
| di_company_mf | str | 위탁 제조사(한글). 대부분(637행) 빈 문자열입니다. |
| di_company_mf_en | str | 위탁 제조사(영문). 대부분(724행) 빈 문자열입니다. |
| di_class_no | str | 약효 분류(예: `[02180]동맥경화용제`)입니다. |
| di_etc_otc_code | str | 의약품 구분. `전문의약품` 또는 `일반의약품`입니다. |
| di_item_permit_date | str | 품목 허가일(YYYYMMDD)입니다. |

### 약 외형

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| dl_custom_shape | str | 제형 분류(예: `정제, 저작정`, `경질캡슐제`)입니다. |
| form_code_name | str | 제형 이름(예: `필름코팅정`, `나정`)입니다. |
| drug_shape | str | 알약 모양(예: `원형`, `타원형`, `장방형`)입니다. |
| chart | str | 성상. 알약 생김새를 글로 설명한 것입니다. |
| color_class1 | str | 주 색상입니다. |
| color_class2 | str | 보조 색상. 대부분 빈 문자열입니다. |
| thick | float | 두께(mm). 0.0은 값이 없는 경우입니다. |
| leng_long | float | 장축 길이(mm). 0.0은 값이 없는 경우입니다. |
| leng_short | float | 단축 길이(mm). 0.0은 값이 없는 경우입니다. |
| print_front | str | 앞면 각인 문자(예: `R5`)입니다. |
| print_back | str | 뒷면 각인 문자. `분할선`처럼 적힌 경우도 있습니다. |
| line_front | str | 앞면 분할선. `-`(일자), `+`(십자), 빈 문자열(없음)입니다. |
| line_back | str | 뒷면 분할선. 값은 `line_front`와 같은 방식입니다. |

### 마크 정보

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| mark_code_front_anal | str | 앞면 마크 설명(예: `pfizer`). 대부분 빈 문자열입니다. |
| mark_code_front_img | str | 앞면 마크 이미지 URL. 대부분 빈 문자열입니다. |
| mark_code_back_anal | str | 뒷면 마크 설명. 전부 비어 있습니다. |
| mark_code_back_img | str | 뒷면 마크 이미지 URL. 전부 비어 있습니다. |
| mark_code_front | str | 앞면 마크 코드. 전부 비어 있습니다. |
| mark_code_back | str | 뒷면 마크 코드. 전부 비어 있습니다. |

### 기타

| 컬럼명 | 데이터 타입 | 설명 |
|---|---|---|
| img_key | str | 공공 낱알 이미지 URL입니다. |
| img_regist_ts | str | 낱알 이미지 등록일(YYYYMMDD)입니다. |
| change_date | str | 약 정보 변경일(YYYYMMDD)입니다. |

> 참고: `img_key`, `chart`, `drug_shape`, `print_*`, `color_*`, `line_*`, `mark_*`, `img_regist_ts`,
> `change_date`, `form_code_name`은 18행이 결측(NaN)입니다. 키는 있고 값이 `null`입니다(4.3에서 확인).
> 원본 kwon 정의서와 dongwook 셀 9는 이를 "키 없음"으로 적었습니다.

## 3. 데이터 규모와 구조

> 출처: JJAM99_eda.ipynb 셀 3–17, 19 (potato) 기준. sohee_eda.ipynb 셀 7, 9, 11 (bravedolphinhere)의
> image_id 대응 확인, 조합별 예시 이미지, 디코딩 실패·SHA256·train/test 공통 해시를 합쳤습니다.

### 3.1 파일 수, 해상도, 색상 모드, 디코딩 오류

> 출처: JJAM99_eda.ipynb 셀 3, 5, 19 (potato) 기준. sohee_eda.ipynb 셀 11 (bravedolphinhere)의
> 디코딩 실패·SHA256을 합쳤습니다.

먼저 이미지 헤더만 읽어 해상도와 색상 모드를 봅니다(빠름). 헤더가 정상이어도 픽셀 손상까지 확인된
것은 아니므로, 전체 디코딩과 SHA256은 다음 셀에서 `RUN_HEAVY`일 때만 계산합니다.
JSON에 적힌 `width`·`height`와 실제 이미지 크기도 라벨마다 대조합니다.

In [ ]:
img_paths = {"train": list_images(TRAIN_IMG_DIR), "test": list_images(TEST_IMG_DIR)}
header_df = pd.concat(
    [read_image_headers(paths, split) for split, paths in img_paths.items()], ignore_index=True
)
header_ok = header_df["error"].isna()
header_df["resolution"] = [
    f"{int(w)}×{int(h)}" if ok else None
    for w, h, ok in zip(header_df["width"], header_df["height"], header_ok, strict=True)
]
header_summary = header_df.groupby("split", sort=False).agg(
    n_images=("file_name", "size"),
    resolution=("resolution", lambda s: sorted(s.dropna().unique())),
    mode=("mode", lambda s: sorted(s.dropna().unique())),
    n_errors=("error", lambda s: s.notna().sum()),
)
print("[원본] 이미지 헤더")
display(
    header_summary.rename(
        columns={
            "n_images": "이미지 수",
            "resolution": "해상도",
            "mode": "색상 모드",
            "n_errors": "헤더 읽기 오류",
        }
    )
)
if not header_ok.all():
    display(header_df[~header_ok])

# 라벨마다 JSON의 width·height와 실제 이미지 크기를 비교한다. 헤더를 못 읽은 이미지는 위에서 셌다.
train_header = header_df[(header_df["split"] == "train") & header_ok]
size_check = ann_df[["source_json", "file_name", "width", "height"]].merge(
    train_header[["file_name", "width", "height"]], on="file_name", suffixes=("_json", "_actual")
)
size_mismatch = size_check[
    (size_check["width_json"] != size_check["width_actual"])
    | (size_check["height_json"] != size_check["height_actual"])
]
print(f"[원본] JSON 크기와 실제 이미지 크기가 다른 라벨: {len(size_mismatch)}개")
if not size_mismatch.empty:
    display(size_mismatch)

In [ ]:
# [RUN_HEAVY] 전체 디코딩·SHA256. 입력 지문·계산 버전이 저장 당시와 같은 캐시만 재사용한다.
INTEGRITY_VERSION = 1  # inspect_image_integrity의 계산 방식을 바꾸면 올린다.
integrity_cache = CACHE_DIR / "image_integrity.json"
integrity_key = {
    "version": INTEGRITY_VERSION,
    "data_state": "원본",
    "inputs": fingerprint_files([*img_paths["train"], *img_paths["test"]], RAW_DIR),
}
if RUN_HEAVY:
    start = time.perf_counter()
    integrity_df = pd.concat(
        [inspect_image_integrity(paths, split) for split, paths in img_paths.items()],
        ignore_index=True,
    )
    integrity_status = f"실행 ({time.perf_counter() - start:.0f}초)"
    if SAVE_OUTPUTS:
        save_cache(integrity_cache, integrity_key, integrity_df)
else:
    integrity_df, integrity_status = load_cache(integrity_cache, integrity_key)
    if integrity_df is None:
        integrity_status = f"{NOT_RUN} ({integrity_status})"

# integrity_df가 None이면 미실행이다. 뒤 셀은 None을 확인하고, 0건으로 바꿔 세지 않는다.
print(f"[원본] 전체 디코딩·SHA256: {integrity_status}")
if integrity_df is not None:
    decoded = integrity_df["decoded"].astype(bool)
    display(
        integrity_df.assign(failed=~decoded)
        .groupby("split", sort=False)
        .agg(n_images=("file_name", "size"), n_failed=("failed", "sum"))
        .rename(columns={"n_images": "이미지 수", "n_failed": "디코딩 실패"})
    )
    if not decoded.all():
        display(integrity_df[~decoded])

### 3.2 1 사진 : N JSON 구조와 bbox 시각 확인

> 출처: JJAM99_eda.ipynb 셀 7–14 (potato) 기준. sohee_eda.ipynb 셀 7, 9 (bravedolphinhere)의
> image_id 대응 확인과 여러 장 예시를 합쳤습니다.

- 사진과 JSON은 JSON 안의 `images[].file_name`으로 연결합니다. JSON 파일명이나 전역 `image_id`로
  합치지 않습니다. 그래도 둘이 사진과 어떻게 대응하는지는 확인해 둡니다.
- 학습할 때는 같은 사진을 가리키는 JSON의 라벨을 모두 모아야 합니다.

In [ ]:
# 사진당 JSON 수와 bbox 수. JSON 하나에 bbox가 하나씩이면 대각선에만 값이 있다.
print("[원본] 사진당 JSON 수 × bbox 수 (값: 사진 수)")
display(
    pd.crosstab(img_df["n_json"], img_df["n_boxes"]).rename_axis(index="JSON 수", columns="bbox 수")
)
boxes_per_json = ann_df.groupby("source_json").size()
n_missing_image = ann_df.loc[~ann_df["file_name"].isin(img_df["file_name"]), "file_name"].nunique()
print(f"[원본] bbox가 2개 이상인 JSON: {(boxes_per_json > 1).sum()}개")
print(f"[원본] JSON이 없는 train 이미지: {(img_df['n_json'] == 0).sum()}장")
print(f"[원본] JSON이 가리키지만 디스크에 없는 이미지: {n_missing_image}장")

# 연결 기준이 아닌 두 값도 사진과 1:1인지 본다. 1:1이어도 연결에는 file_name을 쓴다.
ids_per_file = ann_df.groupby("file_name")["image_id"].nunique()
files_per_id = ann_df.groupby("image_id")["file_name"].nunique()
print(
    f"[원본] image_id가 2개 이상인 사진: {(ids_per_file > 1).sum()}장 / "
    f"사진이 2장 이상인 image_id: {(files_per_id > 1).sum()}개"
)
json_stem = ann_df["source_json"].str.rsplit("/", n=1).str[-1].str.removesuffix(".json")
img_stem = ann_df["file_name"].str.rsplit(".", n=1).str[0]
print(f"[원본] JSON 파일명이 이미지 파일명과 다른 라벨: {(json_stem != img_stem).sum()}개")

In [ ]:
# bbox 수가 다른 사진을 한 장씩 골라 그 사진의 JSON 라벨을 모두 그린다.
sample_names = (
    img_df[img_df["n_boxes"] > 0]
    .drop_duplicates("n_boxes")
    .sort_values("n_boxes")["file_name"]
    .tolist()
)
fig, axes = image_grid(len(sample_names), cols=4)
for ax, name in zip(axes, sample_names, strict=True):
    boxes = ann_df[ann_df["file_name"] == name]
    boxes = boxes.assign(label=boxes["category_id"].astype(str) + " " + boxes["dl_name"])
    title = f"[원본] {name}\nJSON {boxes['source_json'].nunique()}개 / bbox {len(boxes)}개"
    show_image(ax, TRAIN_IMG_DIR / name, title, fontsize=7)
    draw_bboxes(ax, boxes, label_col="label")
plt.tight_layout()
plt.show()

- JSON 하나에는 사진 1장과 그 사진 속 알약 1개의 라벨이 들어 있습니다. 같은 사진의 라벨이
  약 코드별 폴더에 흩어져 있으므로 사진 단위로 모아야 합니다(JJAM 셀 8).
- JSON 파일명은 이미지 파일명과 같아서, 파일명이 같은 JSON이 사진 수만큼 묶입니다. 이것은 구조이지
  중복이 아닙니다(JJAM 셀 17). 내용 중복은 3.3에서 따로 봅니다.
- bbox가 2개인 사진 7장은 모두 파일명에 약이 3개인 사진입니다. 하나씩 라벨이 빠진 것으로,
  4.2에서 봅니다. bbox 3개 사진 중 1장도 파일명에는 약이 4개입니다(원본 상태 관측값).
- 그림의 박스는 JSON에 적힌 좌표입니다. 클래스 ID가 표시된다고 종류까지 맞다는 뜻은 아니며,
  그림만으로 라벨이 빠짐없이 있는지 확정하지 않습니다(라벨 누락은 4.2).

### 3.3 JSON 내용·라벨·픽셀 중복, train/test 중복

> 출처: JJAM99_eda.ipynb 셀 16, 19 (potato) 기준. sohee_eda.ipynb 셀 11 (bravedolphinhere)의
> train/test 공통 SHA256을 합쳤습니다.

파일명이 같다는 이유만으로 중복으로 보지 않습니다. 아래를 따로 셉니다.

| 검사 | 같다고 보는 기준 | 같으면 |
|---|---|---|
| JSON 내용 중복 | 키 순서·공백을 무시한 JSON 전체 | 같은 라벨 파일이 두 번 들어 있음 |
| 라벨 중복 | 사진·`category_id`·bbox | 같은 알약을 두 번 셈 |
| train/test 같은 파일명 | 파일명 | 같은 사진일 수 있음 |
| 파일·픽셀 중복 [RUN_HEAVY] | 파일 SHA256, RGB 픽셀 SHA256 | 같은 사진이 두 번 들어 있음. train/test 사이면 누수 |

회전·재촬영처럼 비슷하기만 한 사진은 8.2(유사도)에서, 다른 약이 같은 좌표에 붙은 경우는 5.2에서 봅니다.

In [ ]:
# JSON 내용 중복: 키 순서·공백을 무시하고 전체 내용이 같은 JSON을 묶는다.
json_hashes = {}
for json_path in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    try:
        data = json.loads(json_path.read_text(encoding="utf-8-sig"))
    except (OSError, UnicodeDecodeError, ValueError):
        continue  # 읽기 실패는 2장의 load_error_df에 있다.
    normalized = json.dumps(data, sort_keys=True, ensure_ascii=False)
    source_json = json_path.relative_to(TRAIN_ANN_DIR).as_posix()
    json_hashes[source_json] = hashlib.sha256(normalized.encode()).hexdigest()
json_hash = pd.Series(json_hashes, name="content_sha256")
dup_json = json_hash[json_hash.duplicated(keep=False)].sort_values()

# 라벨 중복: 같은 사진에 같은 클래스·같은 bbox가 두 번 이상 있는 경우
label_key = ["file_name", "category_id", "x", "y", "w", "h"]
dup_labels = ann_df[ann_df.duplicated(label_key, keep=False)].sort_values(label_key)
same_name = sorted(set(img_df["file_name"]) & set(test_files))


def count_hash_groups(df: pd.DataFrame, col: str) -> dict[str, int]:
    """해시가 같은 이미지 묶음 수를 train 안, test 안, train/test 사이로 나눠 센다."""
    splits = df.dropna(subset=[col]).groupby(col)["split"].agg(list)
    return {
        "train 안": int(splits.map(lambda s: s.count("train") > 1).sum()),
        "test 안": int(splits.map(lambda s: s.count("test") > 1).sum()),
        "train/test 사이": int(splits.map(lambda s: {"train", "test"} <= set(s)).sum()),
    }


dup_rows = [
    ("JSON 내용 중복 (묶음)", dup_json.nunique(), "완료"),
    ("라벨 중복: 사진·클래스·bbox (묶음)", dup_labels.groupby(label_key).ngroups, "완료"),
    ("train/test 같은 파일명 (장)", len(same_name), "완료"),
]
for col, label in [("file_sha256", "파일 SHA256"), ("pixel_sha256", "픽셀 SHA256")]:
    if integrity_df is None:
        groups = dict.fromkeys(["train 안", "test 안", "train/test 사이"], NOT_RUN)
    else:
        groups = count_hash_groups(integrity_df, col)
    for scope, value in groups.items():
        dup_rows.append((f"{label} 중복: {scope} (묶음)", value, integrity_status))
dup_check_df = pd.DataFrame(dup_rows, columns=["검사", "결과", "상태"])

print("[원본] 중복 검사")
display(dup_check_df)
for title, df in [("JSON 내용 중복", dup_json), ("라벨 중복", dup_labels)]:
    if not df.empty:
        print(title)
        display(df)

- JSON 내용 중복과 같은 사진·클래스·bbox의 라벨 중복은 없습니다(JJAM 셀 16, 19 관측과 같음).
- train/test 사이에 같은 파일명은 없습니다. 파일·픽셀이 완전히 같은 사진도 기존 관측에서는 없었습니다
  (JJAM 셀 19: 동일 픽셀 0묶음, sohee 결과: 학습/테스트 공통 SHA256 0). `RUN_HEAVY=False`이고 캐시가
  없으면 위 표에 `미실행`으로 나오며, 이 경우 중복이 없다고 확인된 것이 아닙니다.
- 완전 중복이 없다고 누수가 없는 것은 아닙니다. 같은 알약 조합을 각도만 바꿔 찍은 사진은 8장에서 봅니다.

## 4. 결측

> 출처: dongwook_eda.ipynb 셀 5–9 (def-uk), eda_kwon.ipynb 셀 4–8, 11–12 (gwon753-dev),
> JJAM99_eda.ipynb 셀 32–37 (potato), eda_sohee.ipynb 셀 6–14 (bravedolphinhere)

결측을 세 층으로 나눕니다.

| 절 | 무엇이 비었나 | 학습 영향 |
|---|---|---|
| 4.1 bbox 값 결측 | 라벨은 있는데 좌표·클래스 값이 없음 | 그 라벨을 쓸 수 없음 |
| 4.2 라벨 누락 | 사진에 있는 알약의 라벨 자체가 없음 | 빠진 알약이 배경으로 학습됨 |
| 4.3 메타데이터 결측 | 약 설명 키(각인, 색상, 마크 등)가 비어 있음 | bbox·클래스 학습에는 직접 영향 없음 |

### 4.1 bbox 값 결측

> 출처: dongwook_eda.ipynb 셀 6 (def-uk)의 값 결측 검사 기준. JJAM99_eda.ipynb 셀 33 (potato)의
> bbox 내부 결측 검사를 합쳤습니다.

로더는 bbox가 비었거나 숫자 4개가 아닌 라벨을 빼지 않고 `x, y, w, h`를 NaN으로 둡니다.
크기 0 이하, 이미지 밖 좌표, `area` 불일치처럼 값은 있지만 틀린 경우는 5장에서 봅니다.

In [ ]:
bbox_cols = ["x", "y", "w", "h", "area", "category_id", "category_name"]
print("[원본] bbox·클래스 컬럼별 결측 수")
display(ann_df[bbox_cols].isna().sum().rename("결측 수").to_frame().T)
n_category_link = (load_error_df["stage"] == "category_link").sum()
print(f"[원본] categories에서 category_id를 찾지 못한 라벨: {n_category_link}개")

bbox_missing = ann_df[ann_df[bbox_cols].isna().any(axis=1)]
if not bbox_missing.empty:
    display(bbox_missing[["source_json", "annotation_id", *bbox_cols]])

### 4.2 라벨 누락 (파일명에 있는 약의 bbox 없음)

> 출처: eda_kwon.ipynb 셀 4–8, 11–12 (gwon753-dev) 기준. dongwook_eda.ipynb 셀 6 (def-uk)의
> 같은 검사로 결과를 대조했습니다. kwon 셀 11의 약별 누락 비율 표는 손으로 쓴 표 대신 계산합니다.

파일명 앞부분(`K-003351-018357-036637`)에는 사진에 찍힌 약 코드가 들어 있습니다. 파일명에는 있는데
bbox가 없는 약을 라벨 누락으로 봅니다. 반대로 파일명에 없는 약의 bbox가 있는지도 확인합니다.

- 초록 박스: 라벨된 bbox (숫자는 `category_id`)
- 제목의 `누락`: bbox가 빠진 약 코드와 약 이름

In [ ]:
# 파일명의 약 코드(K-003351 → 3351)와 실제로 라벨된 category_id를 사진마다 비교한다.
label_check = compare_name_codes(img_df, ann_df)
missing_df = label_check[label_check["missing_codes"].str.len() > 0].reset_index(drop=True)
extra_df = label_check[label_check["extra_codes"].str.len() > 0].reset_index(drop=True)

print(
    f"[원본] 라벨이 빠진 이미지: {len(missing_df)}장 / {len(img_df)}장, "
    f"빠진 bbox: {missing_df['missing_codes'].str.len().sum()}개 "
    f"(그 사진들에 라벨된 bbox: {missing_df['n_boxes'].sum()}개)"
)
print(f"[원본] 파일명에 없는 약의 bbox가 있는 이미지: {len(extra_df)}장")
display(missing_df.drop(columns="extra_codes"))
if not extra_df.empty:
    display(extra_df.drop(columns="missing_codes"))

# 약별 누락 비율 = 빠진 bbox ÷ (라벨된 bbox + 빠진 bbox)
missing_by_drug = (
    missing_df["missing_codes"]
    .explode()
    .astype(int)
    .value_counts()
    .rename_axis("category_id")
    .rename("n_missing")
    .reset_index()
    .merge(cat_df[["category_id", "dl_name", "n_boxes"]], on="category_id", how="left")
)
missing_by_drug["누락 비율(%)"] = (
    100 * missing_by_drug["n_missing"] / (missing_by_drug["n_boxes"] + missing_by_drug["n_missing"])
).round(1)
print("[원본] 약별 누락")
display(missing_by_drug.rename(columns={"n_missing": "빠진 bbox", "n_boxes": "라벨된 bbox"}))

In [ ]:
# 라벨이 빠진 사진에 라벨된 bbox를 그리고, 제목에 빠진 약을 적는다.
drug_names = dict(zip(cat_df["category_id"], cat_df["dl_name"], strict=True))
if missing_df.empty:
    print("[원본] 라벨이 빠진 이미지가 없습니다.")
else:
    fig, axes = image_grid(len(missing_df), cols=4)
    for ax, row in zip(axes, missing_df.itertuples(), strict=True):
        missing = "\n".join(f"{c} {drug_names.get(c, '?')}" for c in row.missing_codes)
        show_image(ax, TRAIN_IMG_DIR / row.file_name, f"[원본] {row.combo}\n누락: {missing}")
        draw_bboxes(ax, ann_df[ann_df["file_name"] == row.file_name], color="green")
    plt.tight_layout()
    plt.show()

사진 전체의 라벨이 없는 것이 아니라, 사진 속 알약 하나의 bbox만 빠져 있습니다(kwon 셀 8).
그대로 학습하면 다음 문제가 생길 수 있습니다.

1. 빠진 알약을 배경으로 학습합니다. 모델이 알약을 덜 검출하는(미탐) 쪽으로 기울 수 있습니다.
2. 같은 약의 정답이 헷갈립니다. 예를 들어 3351은 다른 사진에서는 알약으로 라벨되어 있는데,
   누락된 사진에서는 배경으로 취급됩니다. 그 약에 대한 확신이 낮아질 수 있습니다.
3. 이 사진이 validation에 들어가면, 모델이 빠진 알약을 맞게 찾아도 오답(false positive)으로 채점됩니다.
4. 사진을 섞는 증강(mosaic 등)이 bbox 없는 알약 조각을 여러 학습 이미지로 퍼뜨립니다.

**처리 방안 (결정 전, kwon 셀 12)**

1. 그대로 쓰기
2. 빠진 bbox를 직접 그려 넣기 (kwon의 `added_annotations.csv`)
3. 해당 사진을 전부 빼기 (같은 사진의 멀쩡한 bbox도 함께 잃음)
4. 최소한 이 사진들은 validation에서 빼기

이 노트북은 누락을 찾고 보여 주기만 합니다. 보정 기록의 형식과 적용 방식은 5.8에서 정리합니다.

### 4.3 메타데이터 결측

> 출처: dongwook_eda.ipynb 셀 8–9 (def-uk) 기준. JJAM99_eda.ipynb 셀 33, 35, 37 (potato)의 결측 유형
> 구분·null 레코드 집계·null 시각화, eda_sohee.ipynb 셀 9–14 (bravedolphinhere)의 빈 문자열 집계와
> 결측 이유 표를 합쳤습니다.

- 원본 JSON을 다시 읽어 `data/README.md` 스키마의 모든 키를 레코드마다 검사합니다. 로더(`ann_df`)는
  키 없음과 null을 모두 NaN으로 만들기 때문입니다. 함수는 `scan_missing_values`입니다.
- 결측 유형: `키 없음`, `null`, `빈 문자열`(공백만 있는 값 포함), `빈 배열/객체`, `NaN/무한대`,
  `bbox 내부 결측`. 0과 False는 결측으로 세지 않습니다(크기 0은 아래에서 따로 봅니다).
- `필수여부`는 스키마 기준입니다(M 필수, O 선택, C 조건부). O·C 키는 비어 있어도 정상일 수 있습니다.

In [ ]:
missing_scan_df = scan_missing_values(TRAIN_ANN_DIR, schema_df)
kinds = ["키 없음", "null", "빈 문자열", "빈 배열/객체", "NaN/무한대", "bbox 내부 결측"]
missing_table = pd.crosstab(missing_scan_df["key"], missing_scan_df["kind"]).reindex(
    columns=kinds, fill_value=0
)
n_checked = missing_scan_df.groupby("key").size()
missing_table.insert(0, "검사 수", n_checked)
missing_table["결측 비율(%)"] = (100 * missing_table[kinds].sum(axis=1) / n_checked).round(1)
missing_table = missing_table.join(schema_df.set_index("key")[["required", "description"]])
missing_table = missing_table[missing_table["결측 비율(%)"] > 0].sort_values(
    ["결측 비율(%)", "null"], ascending=False
)

# 학습에 바로 쓰는 키. 여기가 비면 라벨을 연결하거나 bbox를 만들 수 없다.
core_keys = [
    "images.id",
    "images.file_name",
    "images.width",
    "images.height",
    "annotations.id",
    "annotations.image_id",
    "annotations.category_id",
    "annotations.bbox",
    "categories.id",
]
core_missing = missing_scan_df[
    missing_scan_df["key"].isin(core_keys) & (missing_scan_df["kind"] != "값 있음")
]
print(
    f"[원본] 스키마 키 {n_checked.size}개 중 결측이 있는 키: {len(missing_table)}개 "
    f"(필수 M: {(missing_table['required'] == 'M').sum()}개)"
)
print(f"[원본] 학습에 쓰는 핵심 키의 결측: {len(core_missing)}건")
with pd.option_context("display.max_rows", 100):
    display(missing_table.rename(columns={"required": "필수여부", "description": "스키마 설명"}))

In [ ]:
# null이 있는 images 레코드가 어떤 약인지 본다.
# 한 약의 레코드가 모두 null이면 약 정보 자체가 빈 것이다.
null_scan = missing_scan_df[
    (missing_scan_df["section"] == "images") & (missing_scan_df["kind"] == "null")
]
null_records = (
    null_scan.groupby(["source_json", "record_index", "file_name", "drug_N"])["key"]
    .agg(n_null="size", null_keys=lambda s: ", ".join(k.removeprefix("images.") for k in s))
    .reset_index()
)
print(
    f"[원본] null 키가 있는 images 레코드: {len(null_records)}개, "
    f"사진 {null_records['file_name'].nunique()}장"
)
null_by_drug = null_records.groupby("drug_N").agg(
    null_records=("source_json", "size"), n_null_keys=("n_null", "max")
)
drug_info = ann_df.groupby("drug_N").agg(dl_name=("dl_name", "first"), all_records=("x", "size"))
null_by_drug = null_by_drug.join(drug_info)
print("[원본] 약별 null 레코드 수 / 그 약의 전체 레코드 수")
display(null_by_drug)
print("[원본] null인 키 목록 (중복 제거)")
display(null_records["null_keys"].value_counts().rename("레코드 수").to_frame())

In [ ]:
# 0은 결측으로 세지 않았지만, 크기(mm)의 0.0은 값이 없다는 뜻으로 쓰였다(kwon 정의서).
size_cols = ["thick", "leng_long", "leng_short"]
size_zero = ann_df[size_cols].eq(0)
print("[원본] 크기(mm)가 0인 bbox 수")
display(size_zero.sum().rename("0인 bbox 수").to_frame().T)
display(
    ann_df.loc[size_zero.any(axis=1)]
    .groupby(["drug_N", "dl_name", *size_cols])
    .size()
    .rename("bbox 수")
    .reset_index()
)

In [ ]:
# null 레코드의 알약(빨강)에도 bbox와 클래스가 연결되어 있는지 본다. 조합이 다른 사진 최대 4장.
null_examples = (
    null_records.assign(combo=null_records["file_name"].str.split("_").str[0])
    .drop_duplicates("combo")
    .head(4)
)
if null_examples.empty:
    print("[원본] null 레코드가 없습니다.")
else:
    fig, axes = image_grid(len(null_examples), cols=4)
    for ax, row in zip(axes, null_examples.itertuples(), strict=True):
        boxes = ann_df[ann_df["file_name"] == row.file_name]
        is_null = boxes["source_json"] == row.source_json
        title = f"[원본] {row.combo}\n빨강: {row.drug_N} (null 키 {row.n_null}개)"
        show_image(ax, TRAIN_IMG_DIR / row.file_name, title)
        draw_bboxes(ax, boxes[~is_null], color="gray")
        draw_bboxes(ax, boxes[is_null], color="red")
    plt.tight_layout()
    plt.show()

**결측을 어떻게 읽을까**

빈 값만 보고 "마크 없음", "분할선 없음"으로 확정하지 않습니다. 정보가 원래 없는 것인지 수집되지 않은
것인지는 데이터 설명과 실제 이미지를 함께 봐야 구분할 수 있습니다(eda_sohee 셀 10). 아래는 비어 있는
이유로 생각해 볼 수 있는 것입니다.

| 키 | 비어 있는 이유로 생각해 볼 수 있는 것 |
|---|---|
| `print_front`, `print_back` | 그 면에 식별문자가 없을 수 있음 |
| `line_front`, `line_back` | 분할선이 없을 수 있음 |
| `color_class2` | 두 번째 색이 없는 단색 알약일 수 있음 |
| `mark_code_*` | 마크가 없거나 마크 정보가 수집되지 않았을 수 있음 |
| `di_company_mf`, `di_company_mf_en` | 위탁제조사가 없거나 영문명이 제공되지 않았을 수 있음 |
| `annotations.segmentation` | 이 데이터는 bbox만 라벨링함 (스키마상 선택 O) |

- 빈 값 대부분은 사진이 아니라 JSON의 약 설명이 비어 있는 것입니다. `dl_company_en`처럼 사진으로 확인할
  수 없는 키도 있고, `print_back`처럼 사진과 비교해 볼 수 있는 키도 있습니다(eda_sohee 셀 13).
- `null` 18건은 모두 `K-016548`(가바토파정 100mg)의 레코드이고, 이 약의 레코드 18개 전부입니다.
  키는 있고 값이 `null`이며(키 없음 0건), 크기(mm)도 0입니다. 사진이 아니라 약 정보 하나가 통째로
  비어 있는 것입니다(원본 상태 관측값).
- 위 그림처럼 이 레코드에도 bbox와 클래스는 연결되어 있으므로 메타데이터 결측 자체는 bbox 학습에
  영향이 없습니다(JJAM 셀 37). 다만 `K-001900-016548-019607-033009` 70° 사진에서는 16548 박스가
  다른 약(33009)과 같은 좌표에 붙어 있고, 실제 16548 알약에는 박스가 없습니다. 이것은 결측이 아니라
  좌표 오류로, 5.2에서 봅니다(JJAM 셀 38).
- 메타데이터를 학습 입력으로 쓰지 않는 한 결측을 채울 필요는 없어 보입니다(dongwook 셀 8). 알약 실제
  크기를 쓰는 7.3에서는 크기 0인 약을 빼고 계산합니다.

## 5. 라벨 오류와 이상치

> 출처: JJAM99_eda.ipynb 셀 19–31, 38–40, 45–48, 53–60 (potato), eda_kwon.ipynb 셀 14–23, 54–86
> (gwon753-dev), EDA_jang.ipynb 셀 8–11 (younghoon1215), eda_sohee.ipynb 셀 15–22,
> sohee_eda.ipynb 셀 2, 11 (bravedolphinhere)

값은 있지만 틀렸거나 이상해 보이는 라벨을 찾습니다. 5.1–5.7은 모두 `[원본]` `ann_df`에서 찾기만 하고
고치지 않습니다. 보정 기록을 모아 메모리에서 적용하고 원본과 비교하는 일은 5.8에서 합니다.

| 절 | 무엇을 찾나 | 판정 |
|---|---|---|
| 5.1 | 이미지 밖 좌표, 크기 0 이하 | 오류 |
| 5.2 | 다른 약과 좌표가 같은 박스 | 오류 (어느 쪽이 틀렸는지는 사람이 판정) |
| 5.3 | 알약 없이 배경만 있는 박스 | 오류 |
| 5.4 | 기록된 `area`와 `w×h`가 다름 | 확인 대상 (bbox 오류 확정 아님) |
| 5.5 | 면적·가로세로 비율이 분포에서 먼 박스 | 후보 (오류 아님) |
| 5.6 | 붙은·부서진 알약 | 확인 대상 |
| 5.7 | 알약보다 넉넉하게 그린 박스 (여백) | 추정 `[RUN_HEAVY]` |
| 5.8 | 위 결과 요약, 보정 기록 통합과 원본·보정 비교 | |

아래 셀은 5장 전체가 쓰는 검사를 한 번에 계산합니다. 박스 픽셀 측정(kwon 방식)은 train 사진을
한 장씩 한 번 읽으며 약 10초 걸립니다.

In [ ]:
AREA_TOL = 1.0  # |area − w×h|가 이보다 크면 area 불일치 (JJAM 셀 46)
TOUCH_GAP = 10  # 같은 사진의 두 박스 간격이 이 px 이하이면 쌍으로 본다 (kwon 셀 58)
DUP_IOU = 0.9  # 쌍의 IoU가 이 이상이면 좌표 중복, 미만이면 맞닿은 알약 (kwon 셀 57)
EMPTY_DIFF_MAX = 15  # 박스 안 diff_p90이 이보다 작으면 빈 배경 (kwon 셀 71)

flag_df = bbox_flags(ann_df, area_tol=AREA_TOL)
pair_df = find_box_pairs(ann_df, max_gap=TOUCH_GAP)
start = time.perf_counter()
pixel_df = measure_box_pixels(ann_df, TRAIN_IMG_DIR)
measured = pixel_df["status"] == "측정"
print(f"[원본] 박스 픽셀 측정: {measured.sum()}개 ({time.perf_counter() - start:.0f}초)")
display(pixel_df["status"].value_counts().rename("bbox 수").to_frame().T)

### 5.1 이미지 밖 좌표 (크기 0 이하 포함)

> 출처: JJAM99_eda.ipynb 셀 19, 21–31 (potato) 기준. eda_kwon.ipynb 셀 56 (gwon753-dev),
> eda_sohee.ipynb 셀 18 (bravedolphinhere), EDA_jang.ipynb 셀 9 (younghoon1215)의 같은 검사로 대조했습니다.

박스 일부라도 이미지(`width × height`) 밖에 있거나 너비·높이가 0 이하인 라벨을 찾습니다.
보통 화면은 이미지 범위만 보여 주므로, 이미지 밖 박스는 좌표를 넓혀 그려야 보입니다(JJAM 셀 22).

- 왼쪽: 원본 사진 범위. 이미지 밖 박스는 보이지 않습니다.
- 가운데: 좌표를 넓힌 화면. 초록 점선이 사진 경계, 빨강이 문제 박스, 회색이 같은 사진의 나머지 라벨입니다.
- 오른쪽: 같은 조합을 다른 각도로 찍은 사진의 같은 약 박스(JJAM 셀 30).

In [ ]:
coord_error_df = ann_df[flag_df["out_of_image"] | flag_df["non_positive"]].join(
    flag_df[["out_of_image", "non_positive"]]
)
print(
    f"[원본] 이미지 밖 좌표: {flag_df['out_of_image'].sum()}개, "
    f"크기 0 이하: {flag_df['non_positive'].sum()}개"
)
cols = ["source_json", "annotation_id", "category_id", "dl_name", *BBOX_COLS, "width", "height"]
display(coord_error_df[[*cols, "out_of_image", "non_positive"]])

for row in coord_error_df.itertuples():
    bad_box = (row.x, row.y, row.w, row.h)
    others = ann_df[(ann_df["file_name"] == row.file_name) & (ann_df.index != row.Index)]
    combo = row.file_name.split("_")[0]
    same_drug = ann_df[
        ann_df["file_name"].str.startswith(combo + "_")
        & (ann_df["file_name"] != row.file_name)
        & (ann_df["category_id"] == row.category_id)
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5.5))
    show_image(axes[0], TRAIN_IMG_DIR / row.file_name, f"[원본] 사진 범위 ({row.camera_la}°)")
    draw_bboxes(axes[0], others, color="gray")
    bad_text = [int(v) for v in bad_box]
    title = f"[원본] 넓힌 좌표 (초록 점선: 사진 경계)\n{row.category_id} bbox={bad_text}"
    show_image(axes[1], TRAIN_IMG_DIR / row.file_name, title)
    draw_image_bounds(axes[1], row.width, row.height)
    draw_bboxes(axes[1], others, color="gray")
    draw_bbox(axes[1], bad_box, str(row.category_id))
    zoom_to(axes[1], [(0, 0, row.width, row.height), bad_box], margin=0.03)
    axes[1].axis("on")
    if same_drug.empty:
        axes[2].set_title("같은 조합의 다른 사진 없음")
    else:
        ref = same_drug.iloc[0]
        ref_text = [int(v) for v in ref[BBOX_COLS]]
        title = f"[원본] 비교: {ref['camera_la']}° 사진의 {ref['category_id']}\nbbox={ref_text}"
        show_image(axes[2], TRAIN_IMG_DIR / ref["file_name"], title)
        draw_bboxes(axes[2], same_drug.iloc[[0]], color="lime")
    fig.suptitle(f"{row.file_name} | {row.dl_name}", fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# 새 위치 제안 (JJAM 셀 26): 믿을 수 없는 x는 버리고, y·크기가 비슷한 색상 윤곽을 사진에서 찾는다.
# 메모리에만 두고 5.8에서 미검수 보정 기록으로 다룬다.
shape_proposals = {}
for row in ann_df[flag_df["out_of_image"]].itertuples():
    bbox, note = propose_bbox_by_shape(TRAIN_IMG_DIR / row.file_name, (row.x, row.y, row.w, row.h))
    shape_proposals[(row.source_json, row.annotation_id)] = bbox
    print(f"[원본] {row.category_id} 라벨 {row.annotation_id}: 제안 bbox {bbox} ({note})")

- `K-003351-016262-018357` 75° 사진의 18357(종근당글리아티린연질캡슐) 1개입니다. bbox `[6567, 625, 311, 315]`의
  x가 사진 너비 976을 훨씬 넘습니다. 크기 0 이하는 없습니다. 다섯 노트북이 모두 같은 1개를 찾았습니다(원본 상태 관측값).
- 같은 조합의 다른 각도 사진에서는 18357 박스가 정상 범위에 있습니다. 압축 전 원본 데이터도 같은 값이어서,
  압축 해제 오류가 아니라 annotation 자체가 틀린 것입니다(JJAM 셀 31).
- 사진 속 18357 캡슐에는 박스가 없습니다. 그대로 쓰면 학습 도구에 따라 좌표를 잘라내거나 라벨·이미지를
  건너뛰어, 이 캡슐이 배경으로 학습될 수 있습니다.
- 고치는 값이 둘입니다. kwon은 박스 없는 알약을 찾아 옮겼고(`x=574`, x=567 앞에 6이 붙은 오타로 추정,
  kwon 셀 83), JJAM은 의도된 값을 추측하지 않고 위 셀처럼 색상 윤곽에서 위치를 다시 찾았습니다(JJAM 셀 24–26).
  두 값은 거의 같습니다(kwon `[574, 626, 300, 309]`, JJAM 방식 `[572, 625, 303, 312]`, IoU 약 0.98). 5.8에서 나란히 그립니다.

### 5.2 다른 약과 좌표 중복

> 출처: eda_kwon.ipynb 셀 57–60, 67–68 (gwon753-dev) 기준. EDA_jang.ipynb 셀 9 (younghoon1215)의
> 정답 약 판정, JJAM99_eda.ipynb 셀 38–40 (potato)의 같은 사진 확인을 합쳤습니다.

같은 사진에서 간격이 `TOUCH_GAP`(10px) 이하인 박스 쌍을 찾고 IoU로 나눕니다.

- IoU ≥ `DUP_IOU`(0.9): 두 약에 같은 좌표를 붙인 라벨 오류입니다. 한 알약에 박스가 두 개 붙고,
  다른 알약 하나에는 박스가 없습니다.
- IoU < 0.9: 실제로 맞닿은 알약입니다(5.6).

어느 쪽 박스가 틀렸는지는 계산으로 정할 수 없습니다. 박스 자리의 알약을 같은 약의 다른 사진과 색·모양·각인으로
비교해 사람이 정했습니다(jang 셀 9, kwon 셀 67). 아래 `DUPLICATE_WRONG`이 그 판정입니다.

In [ ]:
# 사람이 판정한 위치가 틀린 쪽 약 (jang 셀 9, kwon 셀 67). 키: 사진, 값: 틀린 박스의 category_id
DUPLICATE_WRONG = {
    "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": 16548,
    "K-003351-020238-031863_0_2_0_2_70_000_200.png": 20238,
    "K-003351-029667-031863_0_2_0_2_70_000_200.png": 3351,
}
dup_pair_df = pair_df[pair_df["iou"] >= DUP_IOU].copy()
dup_pair_df["wrong_category"] = dup_pair_df["file_name"].map(DUPLICATE_WRONG).astype("Int64")
dup_pair_df["right_category"] = [
    pd.NA if pd.isna(wrong) else (b if wrong == a else a)
    for a, b, wrong in zip(
        dup_pair_df["category_id_1"],
        dup_pair_df["category_id_2"],
        dup_pair_df["wrong_category"],
        strict=True,
    )
]
stale = sorted(set(DUPLICATE_WRONG) - set(dup_pair_df["file_name"]))
print(
    f"[원본] 간격 {TOUCH_GAP}px 이하 박스 쌍: {len(pair_df)}개 "
    f"(좌표 중복 {len(dup_pair_df)}개, 맞닿음 {len(pair_df) - len(dup_pair_df)}개)"
)
print(
    f"판정 없는 좌표 중복: {dup_pair_df['wrong_category'].isna().sum()}개, "
    f"쌍이 없어진 판정: {stale}"
)
display(
    dup_pair_df.assign(
        name_1=dup_pair_df["category_id_1"].map(drug_names),
        name_2=dup_pair_df["category_id_2"].map(drug_names),
    )[["file_name", "category_id_1", "name_1", "category_id_2", "name_2", "iou", "wrong_category"]]
)

In [ ]:
# 빨간 점선: 위치가 틀린 박스, 파랑: 그 자리에 실제로 있는 알약의 박스, 회색: 나머지 라벨
if dup_pair_df.empty:
    print("[원본] 좌표 중복이 없습니다.")
else:
    fig, axes = image_grid(len(dup_pair_df), cols=3, cell=(5, 6.8))
    for ax, pair in zip(axes, dup_pair_df.itertuples(), strict=True):
        boxes = ann_df[ann_df["file_name"] == pair.file_name]
        is_wrong = boxes["category_id"] == pair.wrong_category
        is_right = boxes["category_id"] == pair.right_category
        wrong = "판정 없음" if pd.isna(pair.wrong_category) else pair.wrong_category
        title = (
            f"[원본] {pair.file_name.split('_')[0]}\n"
            f"빨간 점선: {wrong} {drug_names.get(pair.wrong_category, '')} (틀림)\n"
            f"파랑: {pair.right_category} {drug_names.get(pair.right_category, '')} (이 자리 알약)"
        )
        show_image(ax, TRAIN_IMG_DIR / pair.file_name, title)
        draw_bboxes(ax, boxes[~is_wrong & ~is_right], color="gray")
        draw_bboxes(ax, boxes[is_right], color="tab:blue")
        draw_bboxes(ax, boxes[is_wrong], label_col=None, color="red", linestyle="--")
    plt.tight_layout()
    plt.show()

- 좌표 중복은 3쌍이고 모두 IoU 1.0, 모두 70° 사진입니다. jang이 처음 찾았고 kwon과 JJAM(16548 사진)이 확인했습니다(원본 상태 관측값).
  - `K-001900-016548-019607-033009`: 16548(가바토파정 100mg) 박스가 33009 자리에 있습니다. 16548 알약(노란 원형)에는 박스가 없습니다.
    이 16548 레코드는 4.3의 null 메타데이터 레코드이기도 합니다(JJAM 셀 38–40).
  - `K-003351-020238-031863`: 20238 박스가 3351 자리에 있습니다.
  - `K-003351-029667-031863`: 3351 박스가 31863 자리에 있습니다.
- 그대로 학습하면 한 알약을 두 클래스로 가르치고, 박스 없는 알약은 배경으로 가르칩니다.
- kwon은 맞는 쪽 박스는 두고 틀린 쪽 박스를 박스 없는 알약 위치로 옮겼습니다(`fixed_annotations.csv`, 5.8).

### 5.3 빈 배경 박스

> 출처: eda_kwon.ipynb 셀 69–73 (gwon753-dev)

박스 안 픽셀이 배경과 얼마나 다른지 잽니다(kwon 셀 69).

- `fill`: 박스 넓이 중 알약 픽셀 비율. 정상이면 0.6 안팎이지만 투명·어두운 캡슐은 알약이 있어도 낮게 나와 판정에 쓰지 않습니다.
- `diff_p90`: 박스 안 픽셀의 배경 대비 색 차이 상위 10% 값. 투명·어두운 알약도 윤곽과 각인 때문에 크게 나오고,
  빈 배경만 0에 가깝습니다. **판정에는 이 값을 씁니다**(`EMPTY_DIFF_MAX` = 15 미만).

이미지 밖 박스(5.1)는 잴 픽셀이 없어 여기서 빼고 셉니다. kwon 셀 71은 이 박스도 빈 배경으로 세서 2개였습니다.

In [ ]:
empty_mask = measured & (pixel_df["diff_p90"] < EMPTY_DIFF_MAX)
empty_df = ann_df[empty_mask].join(pixel_df[["fill", "diff_p90"]])
print(
    f"[원본] 빈 배경 박스: {len(empty_df)}개 / 측정 {measured.sum()}개 "
    f"(diff_p90 < {EMPTY_DIFF_MAX})"
)
print(
    "diff_p90가 가장 낮은 5개:", pixel_df.loc[measured, "diff_p90"].nsmallest(5).round(1).tolist()
)
display(
    empty_df[
        ["source_json", "annotation_id", "category_id", "dl_name", *BBOX_COLS, "fill", "diff_p90"]
    ]
)
display(pixel_df.loc[measured, ["fill", "diff_p90"]].describe().round(3).T)

In [ ]:
# diff_p90가 가장 낮은(배경과 가장 비슷한) 박스를 잘라 본다. 빨강: 빈 배경 판정, 초록: 알약 있음
lowest = pixel_df.loc[measured, "diff_p90"].nsmallest(12).index
fig, axes = image_grid(len(lowest), cols=6, cell=(2.6, 3.2))
for ax, i in zip(axes, lowest, strict=True):
    row = ann_df.loc[i]
    color = "red" if empty_mask[i] else "green"
    status = "빈 배경" if empty_mask[i] else "알약 있음"
    show_image(ax, TRAIN_IMG_DIR / row["file_name"])
    draw_bbox(ax, row[BBOX_COLS], color=color)
    zoom_to(ax, [row[BBOX_COLS]], margin=0.05, limit=(row["width"], row["height"]))
    title = f"{row['category_id']}  diff {pixel_df.loc[i, 'diff_p90']:.1f}\n{status}"
    ax.set_title(title, fontsize=9, color=color)
fig.suptitle(
    f"[원본] 배경과 가장 비슷한 박스 {len(lowest)}개 (빨강: diff_p90 < {EMPTY_DIFF_MAX})",
    x=0.01,
    ha="left",
)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

- 빈 배경 박스는 `K-003351-020014-020238` 90° 사진의 3351 박스 1개입니다(diff_p90 약 7.5). 다음으로 낮은 박스는
  24 이상이라 기준 15가 두 무리를 잘 가릅니다(원본 상태 관측값).
- 실제 3351 알약은 바로 옆에 박스 없이 있습니다(kwon 셀 83 케이스 3). kwon은 이 박스를 그 알약 위치로 옮겼습니다(5.8).

### 5.4 area ≠ w×h

> 출처: JJAM99_eda.ipynb 셀 45–48 (potato) 기준. sohee_eda.ipynb 셀 2, 11 (bravedolphinhere)의
> 같은 검사로 대조했습니다.

JSON의 `area`와 bbox `w × h`의 차이가 `AREA_TOL`(1px²)보다 큰 라벨을 찾습니다. `area`가 분할 영역 넓이일 수도
있어 불일치만으로 bbox 오류를 확정하지 않습니다(JJAM 셀 45). YOLO 형식 라벨은 bbox만 쓰지만, COCO 방식 평가는
`area`로 작은·중간·큰 물체를 나누므로 값이 틀리면 크기별 점수가 달라질 수 있습니다.

In [ ]:
area_df = ann_df[flag_df["area_mismatch"]].copy()
area_df["bbox_area"] = area_df["w"] * area_df["h"]
area_df["area_diff"] = area_df["area"] - area_df["bbox_area"]
area_df["area_diff_pct"] = (100 * area_df["area_diff"] / area_df["bbox_area"]).round(2)
# sohee는 상대 오차 0.1%까지 같다고 봤다(math.isclose(rel_tol=1e-3, abs_tol=1)).
sohee_same = [
    math.isclose(a, b, rel_tol=1e-3, abs_tol=1.0)
    for a, b in zip(area_df["area"], area_df["bbox_area"], strict=True)
]
print(
    f"[원본] area 불일치: {len(area_df)}개 (|차이| > {AREA_TOL}px²), "
    f"sohee 기준(상대 오차 0.1% 초과): {len(area_df) - sum(sohee_same)}개"
)
cols = ["file_name", "annotation_id", "category_id", "dl_name", *BBOX_COLS, "area", "bbox_area"]
display(
    area_df[[*cols, "area_diff", "area_diff_pct"]]
    .assign(sohee_same=sohee_same)
    .rename(columns={"bbox_area": "w×h", "area_diff": "차이", "area_diff_pct": "차이율(%)"})
)

In [ ]:
fig, axes = image_grid(len(area_df), cols=3, cell=(4.5, 4.8))
for ax, (_, row) in zip(axes, area_df.iterrows(), strict=True):
    title = (
        f"[원본] {row['category_id']} {row['dl_name']}\n"
        f"area={row['area']:g}, w×h={row['bbox_area']:g} ({row['area_diff_pct']:+.2f}%)"
    )
    show_image(ax, TRAIN_IMG_DIR / row["file_name"], title)
    draw_bbox(ax, row[BBOX_COLS])
    zoom_to(ax, [row[BBOX_COLS]], margin=0.3, limit=(row["width"], row["height"]))
plt.tight_layout()
plt.show()

In [ ]:
# 잘린 캡슐 (JJAM 셀 47–48): 박스 중앙의 색상 덩어리 전체를 감싸는 bbox를 다시 제안한다.
# 메모리에만 두고 5.8에서 미검수 보정 기록으로 다룬다.
CUT_CAPSULE = (
    "K-003351-019232-029667_json/K-019232/K-003351-019232-029667_0_2_1_2_70_000_200.json",
    5178,
)
capsule_row = ann_df[
    (ann_df["source_json"] == CUT_CAPSULE[0]) & (ann_df["annotation_id"] == CUT_CAPSULE[1])
]
capsule_proposal = None
if len(capsule_row) != 1:
    print(f"[원본] 잘린 캡슐 제안: {NOT_RUN} (대상 라벨 {len(capsule_row)}개)")
else:
    row = capsule_row.iloc[0]
    old_bbox = tuple(int(v) for v in row[BBOX_COLS])
    capsule_proposal, note = propose_bbox_at_center(TRAIN_IMG_DIR / row["file_name"], old_bbox)
    print(
        f"[원본] {row['category_id']} 라벨 {CUT_CAPSULE[1]}: {list(old_bbox)} "
        f"→ 제안 {capsule_proposal} ({note})"
    )
    fig, axes = plt.subplots(1, 2, figsize=(8, 4.8))
    panels = [("[원본] bbox", old_bbox, "red"), ("[제안] JJAM 방식", capsule_proposal, "lime")]
    for ax, (title, bbox, color) in zip(axes, panels, strict=True):
        show_image(ax, TRAIN_IMG_DIR / row["file_name"], f"{title}\n{bbox}")
        if bbox is not None:
            draw_bbox(ax, bbox, color=color)
        boxes = [b for b in (old_bbox, capsule_proposal) if b is not None]
        zoom_to(ax, boxes, margin=0.2, limit=(row["width"], row["height"]))
    fig.suptitle(f"{row['file_name']} | {row['dl_name']}", fontsize=10)
    plt.tight_layout()
    plt.show()

- area 불일치는 6개로 JJAM 셀 46과 같습니다. sohee 결과(5개)와의 차이는 계산 조건 때문입니다. sohee는 상대 오차
  0.1%까지 같다고 봐서 차이가 작은 1개가 빠졌습니다(위 표 `sohee_same`). 원본 상태 관측값입니다.
- `K-003351-019232-029667` 70° 사진의 19232(라벨 5178)는 기록된 area(79112)가 w×h(55912)보다 크고,
  박스가 캡슐 아래쪽을 자릅니다. JJAM은 bbox를 `[527, 211, 224, 323]`으로 키우고 area를 w×h로 맞춘 수정본을
  `data/processed/bbox_area/`에 저장했습니다(JJAM 셀 47–48). 이 노트북은 같은 방법으로 제안만 다시 계산하며,
  결과가 JJAM 값과 같습니다.
- 나머지 5개는 그림에서 박스가 알약을 감싸고 있어 bbox가 아니라 `area` 값만 다른 것으로 보입니다.

### 5.5 면적·가로세로 비율 이상치 후보 (IQR, 오류 아님)

> 출처: JJAM99_eda.ipynb 셀 19, 21–24 (potato) 기준. eda_sohee.ipynb 셀 19–22 (bravedolphinhere)의
> 가로세로 비율 기준을 합쳤습니다.

- 면적 비율 = `w × h ÷ (width × height)`, 가로세로 비율 = `w ÷ h`
- 5.1의 좌표 오류 박스를 뺀 나머지에서 Q1, Q3을 구하고, `Q1 − 1.5×IQR`보다 작거나 `Q3 + 1.5×IQR`보다 크면 후보로 둡니다.
- 정답의 허용 범위가 아니라 눈으로 볼 후보를 추리는 통계 기준입니다. 크기 분포 자체는 6.2에서 봅니다.

In [ ]:
valid = ann_df[
    ~(flag_df["out_of_image"] | flag_df["non_positive"]) & ann_df[BBOX_COLS].notna().all(axis=1)
]
ratio_df = pd.DataFrame(
    {
        "area_ratio": valid["w"] * valid["h"] / (valid["width"] * valid["height"]),
        "aspect": valid["w"] / valid["h"],
    }
)
ratio_names = {"area_ratio": "면적 비율", "aspect": "가로세로 비율"}
bounds = {col: iqr_bounds(ratio_df[col]) for col in ratio_df}
outlier_flags = pd.DataFrame({col: ~ratio_df[col].between(*bounds[col]) for col in ratio_df})

summary = pd.DataFrame(
    {
        "하한": [bounds[c][0] for c in ratio_df],
        "상한": [bounds[c][1] for c in ratio_df],
        "하한 미만": [(ratio_df[c] < bounds[c][0]).sum() for c in ratio_df],
        "상한 초과": [(ratio_df[c] > bounds[c][1]).sum() for c in ratio_df],
        "후보": outlier_flags.sum().to_numpy(),
    },
    index=[ratio_names[c] for c in ratio_df],
)
print(f"[원본] IQR 후보 (검사한 bbox {len(valid)}개, 좌표 오류 제외)")
display(summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, col in zip(axes, ratio_df, strict=True):
    ax.hist(ratio_df[col], bins=40, color="steelblue")
    for value, color in zip(bounds[col], ["orange", "red"], strict=True):
        ax.axvline(value, color=color, ls="--")
    ax.set(
        title=f"[원본] {ratio_names[col]} (점선: IQR 하한·상한)",
        xlabel=ratio_names[col],
        ylabel="bbox 수",
    )
plt.tight_layout()
plt.show()

# 후보가 어떤 약에 몰려 있는지 본다.
by_drug = (
    outlier_flags.join(ann_df[["category_id", "dl_name"]])
    .groupby(["category_id", "dl_name"])[["area_ratio", "aspect"]]
    .sum()
    .join(cat_df.set_index("category_id")["n_boxes"], on="category_id")
)
by_drug = by_drug[by_drug[["area_ratio", "aspect"]].sum(axis=1) > 0].sort_values(
    ["area_ratio", "aspect"], ascending=False
)
print("[원본] 약별 후보 수 (n_boxes: 그 약의 전체 bbox 수)")
display(by_drug.rename(columns=ratio_names))

In [ ]:
# 기준에서 가장 멀리 벗어난 후보를 기준마다 4개씩 본다.
examples = []
for col in ratio_df:
    low, high = bounds[col]
    distance = np.maximum(low - ratio_df[col], ratio_df[col] - high)
    examples += [(col, i) for i in distance[outlier_flags[col]].nlargest(4).index]
fig, axes = image_grid(len(examples), cols=4, cell=(3.5, 4.2))
for ax, (col, i) in zip(axes, examples, strict=True):
    row = ann_df.loc[i]
    value = ratio_df.loc[i, col]
    title = f"[원본] {ratio_names[col]} {value:.3f}\n{row['category_id']} {row['dl_name']}"
    show_image(ax, TRAIN_IMG_DIR / row["file_name"], title)
    draw_bbox(ax, row[BBOX_COLS])
    zoom_to(ax, [row[BBOX_COLS]], margin=0.3, limit=(row["width"], row["height"]))
plt.tight_layout()
plt.show()

- 면적 비율 후보는 24개이고 모두 상한(약 14.2%)을 넘는 큰 박스입니다(JJAM 셀 24, eda_sohee 셀 20과 같은 수).
  가로세로 비율 후보는 82개입니다(eda_sohee 셀 22). 원본 상태 관측값입니다.
- JJAM과 sohee가 후보 그림을 모두 본 결과, 박스가 큰 캡슐이나 길쭉한 알약을 잘 감싸고 있어 크기만으로 잘못된 라벨이라
  볼 근거는 없었습니다(JJAM 셀 23–24). 기울어진 긴 알약은 축에 평행한 bbox 안에 배경이 더 들어갑니다.
- 후보가 몇몇 약에 몰려 있으므로(위 약별 표), 전체 분포가 아니라 같은 약끼리 비교해야 이상한 박스가 드러납니다.
  같은 약끼리 비교하는 검사는 5.6의 `area_ratio`입니다.

### 5.6 붙은·부서진 알약

> 출처: eda_kwon.ipynb 셀 54–66, 84–86 (gwon753-dev)

1. 알약 상태 값: `drug_S`에 정상 외의 값이 있는가
2. 붙어 있는 알약: 같은 사진에서 간격 10px 이하이고 IoU가 0.9 미만인 박스 쌍 (IoU 0.9 이상은 5.2)
3. 모양 이상: 박스 안 알약 픽셀 넓이를 같은 약끼리 비교하고(`area_ratio` = 같은 약 중앙값 대비),
   윤곽이 패였는지(`solidity`), 덩어리가 여럿인지(`n_blobs`) 봅니다.

**주의**: 투명한 캡슐, 어두운 캡슐, 배경과 색이 비슷한 흰 알약은 OpenCV가 알약을 조각내 잡아서 정상이어도
의심으로 나옵니다. 그래서 그림으로 반드시 확인합니다(kwon 셀 61).

In [ ]:
print("[원본] drug_S 값")
display(ann_df["drug_S"].value_counts(dropna=False).rename("bbox 수").to_frame().T)

touch_df = pair_df[pair_df["iou"] < DUP_IOU]
print(f"[원본] 맞닿은 박스 쌍 (간격 ≤ {TOUCH_GAP}px, IoU < {DUP_IOU}): {len(touch_df)}개")
display(touch_df[["file_name", "category_id_1", "category_id_2", "gap_px", "iou"]].round(3))
if not touch_df.empty:
    fig, axes = image_grid(len(touch_df), cols=4)
    for ax, pair in zip(axes, touch_df.itertuples(), strict=True):
        boxes = ann_df[ann_df["file_name"] == pair.file_name]
        is_pair = boxes["category_id"].isin([pair.category_id_1, pair.category_id_2])
        title = (
            f"[원본] {pair.file_name.split('_')[0]} ({boxes['camera_la'].iloc[0]}°)\n"
            f"{pair.category_id_1} ↔ {pair.category_id_2}, IoU {pair.iou:.3f}"
        )
        show_image(ax, TRAIN_IMG_DIR / pair.file_name, title)
        draw_bboxes(ax, boxes[~is_pair], color="gray")
        draw_bboxes(ax, boxes[is_pair], color="red")
    plt.tight_layout()
    plt.show()

In [ ]:
AREA_LOW, AREA_HIGH = 0.7, 1.3  # 같은 약 중앙값 대비 알약 넓이가 이 범위 밖이면 의심
SOLIDITY_MIN = 0.9  # 이보다 낮으면 윤곽이 패인 것(깨짐, 이 빠짐, 두 알 붙음)으로 의심
EMPTY_MAX = 0.1  # 알약 넓이가 같은 약 중앙값의 10% 미만이면 박스 안에 알약이 없는 것

shape_df = ann_df.loc[measured, ["file_name", "category_id", *BBOX_COLS, "width", "height"]].join(
    pixel_df[["pill_area", "solidity", "n_blobs"]]
)
median_area = shape_df.groupby("category_id")["pill_area"].transform("median")
shape_df["area_ratio"] = shape_df["pill_area"] / median_area
reasons = pd.DataFrame(
    {
        "알약 없음(라벨 오류?)": shape_df["area_ratio"] < EMPTY_MAX,
        "작음(부서짐?)": shape_df["area_ratio"].between(EMPTY_MAX, AREA_LOW, inclusive="left"),
        "큼(붙음?)": shape_df["area_ratio"] > AREA_HIGH,
        "윤곽 패임": shape_df["solidity"] < SOLIDITY_MIN,
        "덩어리 여러 개": shape_df["n_blobs"] >= 2,
    }
)
shape_df["reason"] = reasons.apply(lambda r: ", ".join(r.index[r]), axis=1)
shape_suspect_df = shape_df[shape_df["reason"] != ""].sort_values("solidity")

print(
    f"[원본] 모양 검사 bbox: {len(shape_df)}개 / 의심: {len(shape_suspect_df)}개 "
    "(이유는 겹칠 수 있음)"
)
display(reasons.sum().rename("bbox 수").to_frame().T)
display(
    shape_suspect_df[
        ["file_name", "category_id", "area_ratio", "solidity", "n_blobs", "reason"]
    ].round(3)
)

In [ ]:
# 의심 박스(빨간 제목)와 같은 약의 정상 박스(초록 제목)를 나란히 본다. 앞의 12개만 그린다.
shown = shape_suspect_df.head(12)
normal = shape_df[shape_df["reason"] == ""]
fig, axes = image_grid(2 * len(shown), cols=4, cell=(2.8, 3.3))
for i, s in enumerate(shown.itertuples()):
    ax_s, ax_r = axes[2 * i], axes[2 * i + 1]
    show_image(ax_s, TRAIN_IMG_DIR / s.file_name)
    zoom_to(ax_s, [(s.x, s.y, s.w, s.h)], margin=0.02, limit=(s.width, s.height))
    ax_s.set_title(
        f"의심 {s.category_id}\n" + s.reason.replace(", ", "\n"), fontsize=8, color="red"
    )
    same = normal[normal["category_id"] == s.category_id]
    if not same.empty:
        ref = same.loc[(same["area_ratio"] - 1).abs().idxmin()]
        show_image(ax_r, TRAIN_IMG_DIR / ref["file_name"])
        zoom_to(ax_r, [ref[BBOX_COLS]], margin=0.02, limit=(ref["width"], ref["height"]))
        ax_r.set_title(f"정상 예시 {s.category_id}", fontsize=8, color="green")
    else:
        ax_r.set_title(f"{s.category_id}: 정상 판정 박스 없음", fontsize=8)
fig.suptitle(f"[원본] 모양 의심 {len(shown)}개 / 전체 {len(shape_suspect_df)}개", x=0.01, ha="left")
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

- `drug_S`는 모두 `정상알약`입니다.
- 맞닿은 박스 쌍은 `K-003351-033880-035206`의 75°, 90° 사진 2개입니다(같은 장면). IoU 0.02 정도로 살짝 겹칠 뿐
  라벨은 정상입니다(kwon 셀 83). 원본 상태 관측값입니다.
- 모양 의심은 kwon이 라벨 누락을 채운 771개 기준으로 30개였고, 라벨 오류 3개(3351 빈 배경, 18357 이미지 밖,
  16548 다른 약 자리)를 빼면 부서지거나 붙은 알약은 0개, 나머지 27개는 투명·어두운·흰 알약의 오탐이었습니다
  (kwon 셀 65–66).
- 이 노트북은 원본 763개에서 이미지 밖 박스(18357)를 빼고 재서 의심이 29개입니다. 라벨 오류 2개(3351 빈 배경은
  `알약 없음`, 16548 다른 약 자리는 `큼`)를 빼면 27개이고, 모두 약 6종(16688, 2483, 27733, 31885, 27926, 35206)에
  몰려 있어 kwon이 오탐으로 본 유형과 같습니다. 원본 상태 관측값입니다.
- 맞닿은 알약 처리(kwon 셀 85–86에서 정함): 데이터는 그대로 둡니다. 대신 추론 NMS가 클래스별로 적용되는지,
  IoU 기준이 너무 낮지 않은지 확인하고, 같은 장면 2장은 그룹 분할로 한쪽에만 둡니다. 학습 후 붙은 알약을 놓치면
  그때 Copy-Paste 증강을 추가합니다. 9장 "학습 파이프라인으로 넘길 것"에 옮깁니다.

### 5.7 bbox 여백 (GrabCut 추정) `[RUN_HEAVY]`

> 출처: JJAM99_eda.ipynb 셀 53–60 (potato). 함수는 `data_process/data_processing.py`의 `measure_bbox_padding`입니다.

알약마다 GrabCut으로 몸체를 추정하고 `padding_ratio = 1 − 몸체 넓이 ÷ bbox 넓이`를 잽니다.

- 그림자 후보를 배경으로 지정하고 초기 조건 두 가지로 GrabCut을 돌려, 두 윤곽을 모두 포함하는 사각형을 몸체로 봅니다.
  몸체에 최소 4px 여유를 더한 것이 `tight_bbox`(새 bbox 후보)입니다.
- `status`가 `estimated`인 박스만 여백을 집계합니다. 탐색 경계에 닿거나, 두 추정이 다르거나, 윤곽이 원본 박스 밖으로
  이어지면 `review_required`·`extends_outside`로 따로 둡니다.
- 원형이나 기울어진 알약은 축에 평행한 박스 모서리에 빈 공간이 생기는 게 정상이라, 여백 비율만으로 박스가 틀렸다고 보지 않습니다.
- 전체 박스에 1–2분 걸려 `RUN_HEAVY=True`일 때만 계산합니다. 아니면 입력·방식·seed가 같은 캐시만 씁니다.

이식하지 않은 셀: 셀 61–62(이전 색상 전용 방식과 비교), 63–66(보정 전후 개수와 그림), 67–74(보정본 저장, 미분류
재보정). 저장 셀은 계획 6장 원칙에 따라 옮기지 않았고, 여백 20% 이상 후보는 5.8에서 미검수 보정 기록으로 다룹니다.

In [ ]:
# [RUN_HEAVY] GrabCut 여백 추정. 입력 지문·방식·seed가 저장 당시와 같은 캐시만 재사용한다.
padding_cache = CACHE_DIR / "bbox_padding.json"
padding_key = {
    "version": PADDING_VERSION,
    "seed": SEED,
    "data_state": "원본",
    "inputs": fingerprint_files([*img_paths["train"], *TRAIN_ANN_DIR.rglob("*.json")], RAW_DIR),
}
if RUN_HEAVY:
    start = time.perf_counter()
    padding_df = measure_bbox_padding(ann_df, TRAIN_IMG_DIR, seed=SEED)
    padding_status = f"실행 ({time.perf_counter() - start:.0f}초)"
    if SAVE_OUTPUTS:
        save_cache(padding_cache, padding_key, padding_df)
else:
    padding_df, padding_status = load_cache(padding_cache, padding_key)
    if padding_df is None:
        padding_status = f"{NOT_RUN} ({padding_status})"

# padding_df가 None이면 미실행이다. 뒤 셀은 None을 확인하고, 여백 후보 0개로 바꿔 세지 않는다.
print(f"[원본] GrabCut 여백 추정: {padding_status}")
if padding_df is not None:
    estimated = padding_df[padding_df["status"] == "estimated"]
    display(padding_df["status"].value_counts().rename("bbox 수").to_frame().T)
    threshold_df = pd.DataFrame(
        [
            {
                "추정 여백 기준": f"{t:.0%} 이상",
                "bbox 수": (estimated["padding_ratio"] >= t).sum(),
                "사진 수": estimated.loc[estimated["padding_ratio"] >= t, "file_name"].nunique(),
                "클래스 수": estimated.loc[
                    estimated["padding_ratio"] >= t, "category_id"
                ].nunique(),
            }
            for t in (0.1, 0.2, 0.3)
        ]
    ).set_index("추정 여백 기준")
    print(f"[원본] 여백 기준별 bbox (estimated {len(estimated)}개 중, 누적이므로 행을 더하지 않음)")
    display(threshold_df)
    padding_review_df = padding_df[padding_df["status"] != "estimated"]
    print(f"[원본] 경계 확인 필요: {len(padding_review_df)}개")
    display(
        padding_review_df[
            ["file_name", "annotation_id", "category_id", "bbox", "status", "review_reason"]
        ]
    )

In [ ]:
# 여백 10~20%, 20~30%, 30% 이상 구간에서 하한에 가장 가까운 박스를 하나씩 본다(JJAM 셀 56).
# 빨강: 원본 bbox, 초록: 경계 여유를 포함한 추정 bbox.
# 아래 표는 원본·추정 bbox IoU 분포다(JJAM 셀 60).
if padding_df is None:
    print(f"[원본] 여백 예시와 IoU 분포: {padding_status}")
else:
    fig, axes = plt.subplots(1, 3, figsize=(10, 4.2))
    for ax, (low, high) in zip(axes, [(0.1, 0.2), (0.2, 0.3), (0.3, 1.01)], strict=True):
        band = estimated[estimated["padding_ratio"].between(low, high, inclusive="left")]
        if band.empty:
            ax.set_title(f"{low:.0%} 이상: 예시 없음")
            ax.axis("off")
            continue
        ex = band.sort_values(["padding_ratio", "file_name", "annotation_id"]).iloc[0]
        title = (
            f"[원본] 여백 {ex['padding_ratio']:.1%}\n{ex['category_id']}, IoU {ex['bbox_iou']:.3f}"
        )
        show_image(ax, TRAIN_IMG_DIR / ex["file_name"], title)
        draw_bbox(ax, ex["bbox"], color="red")
        draw_bbox(ax, ex["tight_bbox"], color="lime")
        zoom_to(ax, [ex["bbox"], ex["tight_bbox"]], margin=0.15, limit=(976, 1280))
    plt.tight_layout()
    plt.show()

    edges = [i / 10 for i in range(11)]
    labels = [f"{lo:.1f}–{hi:.1f}" for lo, hi in zip(edges[:-1], edges[1:], strict=True)]
    bins = edges[:-1] + [1.000001]
    per_box = pd.cut(estimated["bbox_iou"], bins=bins, labels=labels, right=False)
    per_image = pd.cut(
        estimated.groupby("file_name")["bbox_iou"].min(), bins=bins, labels=labels, right=False
    )
    iou_table = pd.DataFrame(
        {
            "bbox 수": per_box.value_counts(sort=False),
            "사진 수 (사진별 최저 IoU)": per_image.value_counts(sort=False),
        }
    ).rename_axis("원본↔추정 IoU")
    display(iou_table[iou_table.sum(axis=1) > 0])

- 여백 10% 이상 747개, 20% 이상 173개(사진 120장), 30% 이상 7개로 JJAM 셀 54와 같습니다. 원본·추정 IoU 분포도
  JJAM 셀 60과 거의 같습니다(원본 상태 관측값).
- 경계 확인은 9개로 JJAM보다 1개 많습니다. JJAM은 보정본(`bbox_area`, `bbox_range`)이 있으면 그것을 입력으로 써서
  18357이 고쳐진 좌표로 들어갔고, 이 노트북은 원본을 입력으로 써서 18357이 `invalid_bbox`(이미지 밖)로 남습니다.
- 대부분의 박스가 10% 이상 여백을 갖습니다. kwon이 본 것처럼 기존 박스는 알약 윤곽보다 사방 약 8px 넉넉하게
  그려져 있습니다(kwon 셀 16). 여백이 일관되면 모델도 일관되게 배우므로 그 자체로 오류는 아닙니다.
- 20% 이상만 줄이면 다른 박스와 여백 기준이 달라집니다(kwon 셀 85의 같은 지적). 정답 박스의 여백 기준을 바꿀지는
  9장 "결정할 것"으로 넘깁니다.

### 5.8 라벨 오류 요약과 보정 기록 통합

> 출처: eda_kwon.ipynb 셀 14–23, 67–83 (gwon753-dev), JJAM99_eda.ipynb 셀 25–26, 47–48, 63–68 (potato),
> EDA_jang.ipynb 셀 10–11 (younghoon1215). 보정 기록 함수는 `data_process/make_labels.py`에 있습니다.

먼저 4–5장에서 찾은 문제를 한 표로 모읍니다. 건수는 이번 실행의 `[원본]` 값입니다.

In [ ]:
n_padding = NOT_RUN if padding_df is None else int((estimated["padding_ratio"] >= 0.2).sum())
error_summary = pd.DataFrame(
    [
        ("4.1", "bbox 값 결측", len(bbox_missing), "오류", "-"),
        ("4.2", "라벨 누락", missing_df["missing_codes"].str.len().sum(), "오류", "추가 (kwon)"),
        (
            "5.1",
            "이미지 밖 좌표",
            flag_df["out_of_image"].sum(),
            "오류",
            "위치 수정 (kwon, JJAM 값이 다름)",
        ),
        ("5.1", "크기 0 이하", flag_df["non_positive"].sum(), "오류", "-"),
        ("5.2", "다른 약과 좌표 중복 (쌍)", len(dup_pair_df), "오류", "틀린 쪽 위치 수정 (kwon)"),
        ("5.3", "빈 배경 박스", len(empty_df), "오류", "위치 수정 (kwon)"),
        ("5.4", "area ≠ w×h", len(area_df), "확인 대상", "19232만 bbox 확대 제안 (JJAM)"),
        ("5.5", "면적 비율 IQR 후보", outlier_flags["area_ratio"].sum(), "후보", "보정 안 함"),
        ("5.5", "가로세로 비율 IQR 후보", outlier_flags["aspect"].sum(), "후보", "보정 안 함"),
        ("5.6", "맞닿은 알약 (쌍)", len(touch_df), "정상 라벨", "보정 안 함 (NMS·그룹 분할)"),
        ("5.6", "모양 이상 의심", len(shape_suspect_df), "확인 대상", "보정 안 함"),
        ("5.7", "여백 20% 이상", n_padding, "추정", "여백 보정 제안 (JJAM)"),
    ],
    columns=["절", "문제", "[원본] 건수", "판정", "보정 기록"],
).set_index("절")
display(error_summary)

**보정 기록 통합 규칙** (계획 6장)

- 같은 문제를 여러 노트북이 다른 방식으로 고쳤으므로, 모든 보정을 한 형식의 표(`correction_df`, 열은 `CORRECTION_COLS`)로 모읍니다.
  한 행이 보정 하나입니다. 주요 열: `stage`(추가 → 위치 수정 → 여백 보정 순으로 적용), `source_json`·`annotation_id`(대상 라벨),
  `old_bbox`(보정할 때 본 좌표), `new_bbox`, `method`, `author`, `reviewed`, `source_record`(원래 기록), `input_sha256`(원본 JSON 해시).
- kwon CSV는 `(file_name, category_id)`로 라벨을 가리키므로 원본 라벨로 역매핑합니다. 후보가 하나가 아니면 적용하지 않습니다.
- 새 라벨 ID는 원본 ID와 겹치지 않게 1,000,000부터 `correction_id` 순서로 붙입니다. 같은 기록이면 실행마다 같은 ID입니다.
- 적용 조건: 검수됨(`reviewed`), 같은 단계에 같은 대상을 가리키는 기록이 하나뿐, 현재 bbox = `old_bbox`, segmentation 없음,
  새 bbox가 이미지 안이고 크기가 양수. 하나라도 어기면 적용하지 않고 이유를 남깁니다(`미검수`, `거부`).
  적용한 박스는 `area = w×h`로 맞춥니다. 이미 적용된 기록은 건너뛰므로 다시 적용해도 결과가 같습니다.
- 적용은 메모리의 `ann_corrected_df`에만 합니다. `ann_df`와 `data/raw/`는 그대로이고, 이 노트북은 기록 파일을 쓰지 않습니다.

| 이번 실행의 기록 출처 | 내용 | `reviewed` |
|---|---|---|
| kwon `data/processed/added_annotations.csv`, `fixed_annotations.csv` | 라벨 누락 8개 추가, 위치 수정 5개 | `KWON_REVIEWED` (기본 True: kwon이 전후 그림으로 확인, kwon 셀 79–80) |
| JJAM 색상 윤곽 제안 (5.1에서 다시 계산) | 18357 위치 | False |
| JJAM 잘린 캡슐 제안 (5.4에서 다시 계산) | 19232 bbox 확대 | False |
| JJAM GrabCut (5.7) | 여백 20% 이상 박스의 `tight_bbox` | False (자동 추정이며 수동 검수가 아님, JJAM 셀 67) |

JJAM 저장본(`data/processed/bbox_area/`, `bbox_grabcut/`)은 팀에 공유되지 않아 같은 방법으로 다시 계산한 값을 씁니다.

In [ ]:
KWON_ADDED_PATH = PROCESSED_DIR / "added_annotations.csv"
KWON_FIXED_PATH = PROCESSED_DIR / "fixed_annotations.csv"
KWON_REVIEWED = True  # kwon이 전후 그림으로 확인함(kwon 셀 79–80). 팀 검수 기준이 정해지면 바꾼다.
PADDING_MIN = 0.2  # 여백이 이 이상인 박스만 여백 보정 후보로 둔다(JJAM 셀 64)

kwon_records = corrections_from_kwon(
    KWON_ADDED_PATH, KWON_FIXED_PATH, ann_df, TRAIN_ANN_DIR, reviewed=KWON_REVIEWED
)
jjam_rows = []
for (source_json, annotation_id), bbox in shape_proposals.items():
    if bbox is not None:
        jjam_rows.append(
            correction_for_label(
                ann_df,
                source_json,
                annotation_id,
                bbox,
                TRAIN_ANN_DIR,
                correction_id=f"jjam_shape:{source_json}#{annotation_id}",
                stage="위치 수정",
                case="이미지 밖 좌표",
                author="potato",
                reviewed=False,
                method="HSV 색상 윤곽 중 y·크기가 비슷한 것 + 사방 8px",
                source_record="JJAM99_eda.ipynb 셀 26 (5.1에서 다시 계산)",
            )
        )
if capsule_proposal is not None:
    jjam_rows.append(
        correction_for_label(
            ann_df,
            *CUT_CAPSULE,
            capsule_proposal,
            TRAIN_ANN_DIR,
            correction_id=f"jjam_capsule:{CUT_CAPSULE[0]}#{CUT_CAPSULE[1]}",
            stage="위치 수정",
            case="잘린 bbox",
            author="potato",
            reviewed=False,
            method="박스 중앙의 HSV 색상 덩어리 + 사방 8px",
            source_record="JJAM99_eda.ipynb 셀 48 (5.4에서 다시 계산)",
        )
    )
padding_rows = []
if padding_df is not None:
    candidates = estimated[estimated["padding_ratio"] >= PADDING_MIN]
    for r in candidates.itertuples():
        padding_rows.append(
            correction_for_label(
                ann_df,
                r.source_json,
                r.annotation_id,
                r.tight_bbox,
                TRAIN_ANN_DIR,
                correction_id=f"jjam_grabcut:{r.source_json}#{r.annotation_id}",
                stage="여백 보정",
                case="bbox 여백",
                author="potato",
                reviewed=False,
                method=f"GrabCut 몸체 + 경계 여유 ({PADDING_VERSION})",
                source_record="JJAM99_eda.ipynb 셀 54, 64 (5.7에서 다시 계산)",
            )
        )

parts = [kwon_records, pd.DataFrame(jjam_rows + padding_rows, columns=CORRECTION_COLS)]
parts = [p for p in parts if not p.empty]
correction_df = (
    pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=CORRECTION_COLS)
)
correction_df = assign_new_ids(correction_df, ann_df)

kwon_sources = kwon_records["source_record"].str.split("#").str[0]
source_df = pd.DataFrame(
    [
        *[
            (
                p.relative_to(PROJECT_ROOT).as_posix(),
                "읽음" if p.is_file() else "파일 없음",
                (kwon_sources == p.name).sum(),
            )
            for p in (KWON_ADDED_PATH, KWON_FIXED_PATH)
        ],
        ("JJAM 색상 제안 (5.1, 5.4)", "다시 계산", len(jjam_rows)),
        (
            "JJAM GrabCut (5.7)",
            padding_status,
            NOT_RUN if padding_df is None else len(padding_rows),
        ),
    ],
    columns=["출처", "상태", "기록 수"],
)
print(f"보정 기록: {len(correction_df)}개")
display(source_df)
display(
    correction_df[correction_df["stage"] != "여백 보정"].drop(
        columns=["input_sha256", "source_json"]
    )
)

In [ ]:
# 같은 라벨에 값이 다른 기록이 둘 이상이면 나란히 그린다(계획 6장: 두 값을 비교해 하나를 고른다).
fixes = correction_df[correction_df["stage"] != "추가"]
conflict_keys = ["stage", "source_json", "annotation_id"]
conflict_df = fixes[fixes.duplicated(conflict_keys, keep=False)]
if conflict_df.empty:
    print("같은 라벨에 값이 다른 보정 기록이 없습니다.")
else:
    groups = list(conflict_df.groupby(conflict_keys))
    fig, axes = image_grid(len(groups), cols=2, cell=(6, 6.5))
    for ax, ((stage, _, annotation_id), group) in zip(axes, groups, strict=True):
        first = group.iloc[0]
        boxes = ann_df[ann_df["file_name"] == first["file_name"]]
        lines = []
        show_image(ax, TRAIN_IMG_DIR / first["file_name"])
        draw_bboxes(ax, boxes[boxes["annotation_id"] != annotation_id], color="gray")
        styles = [("tab:blue", "-", "실선"), ("tab:orange", "--", "점선"), ("magenta", ":", "점")]
        for (color, ls, name), rec in zip(styles, group.itertuples(), strict=False):
            draw_bbox(ax, rec.new_bbox, color=color, linestyle=ls)
            lines.append(f"{name}: {rec.author} {list(rec.new_bbox)} (reviewed={rec.reviewed})")
        iou = bbox_iou(group["new_bbox"].iloc[0], group["new_bbox"].iloc[1])
        title = f"[제안 비교] {first['file_name'].split('_')[0]} {first['category_id']} ({stage})\n"
        ax.set_title(title + "\n".join(lines) + f"\n두 제안의 IoU {iou:.3f}", fontsize=8)
        size = (boxes["width"].iloc[0], boxes["height"].iloc[0])
        zoom_to(ax, list(group["new_bbox"]), margin=0.8, limit=size)
    plt.tight_layout()
    plt.show()

In [ ]:
ann_corrected_df, apply_log_df = apply_corrections(ann_df, correction_df)
n_applied = int((apply_log_df["status"] == "적용").sum())
print(
    f"[보정] 기록 {len(correction_df)}개 중 적용 {n_applied}개, "
    f"bbox {len(ann_df)}개 → {len(ann_corrected_df)}개"
)
status_table = pd.crosstab(apply_log_df["stage"], apply_log_df["status"])
display(status_table.reindex([s for s in STAGES if s in status_table.index]))
not_applied = apply_log_df[apply_log_df["status"] != "적용"]
print("[보정] 적용하지 않은 기록 (이유별)")
display(not_applied.groupby(["stage", "status", "reasons"]).size().rename("기록 수").reset_index())

# 같은 기록을 보정본에 다시 적용해도 새로 적용되는 것이 없어야 한다(중복 추가 방지).
again_df, again_log = apply_corrections(ann_corrected_df, correction_df)
print(
    f"[보정] 다시 적용: 새로 적용 {(again_log['status'] == '적용').sum()}개, "
    f"이미 적용 {(again_log['status'] == '이미 적용').sum()}개, "
    f"bbox 수 변화 {len(again_df) - len(ann_corrected_df)}개"
)

In [ ]:
# 원본과 보정본을 같은 검사로 비교한다. 바뀐 박스만 픽셀을 다시 잰다.
correction_status = "완료" if n_applied else f"{NOT_RUN} (적용된 보정 없음)"
print(f"[보정] 원본·보정 비교: {correction_status}")
if n_applied:
    changed = ann_corrected_df["correction_ids"] != ""
    flags_after = bbox_flags(ann_corrected_df, area_tol=AREA_TOL)
    pairs_after = find_box_pairs(ann_corrected_df, max_gap=TOUCH_GAP)
    diff_after = pixel_df["diff_p90"].reindex(ann_corrected_df.index)
    diff_after[changed] = measure_box_pixels(ann_corrected_df[changed], TRAIN_IMG_DIR)["diff_p90"]
    check_after = compare_name_codes(img_df, ann_corrected_df)
    compare_df = pd.DataFrame(
        [
            ("bbox 수", len(ann_df), len(ann_corrected_df)),
            (
                "라벨 누락 (bbox)",
                label_check["missing_codes"].str.len().sum(),
                check_after["missing_codes"].str.len().sum(),
            ),
            (
                "파일명에 없는 약의 bbox",
                label_check["extra_codes"].str.len().sum(),
                check_after["extra_codes"].str.len().sum(),
            ),
            ("이미지 밖 좌표", flag_df["out_of_image"].sum(), flags_after["out_of_image"].sum()),
            ("크기 0 이하", flag_df["non_positive"].sum(), flags_after["non_positive"].sum()),
            ("area ≠ w×h", flag_df["area_mismatch"].sum(), flags_after["area_mismatch"].sum()),
            ("좌표 중복 쌍", len(dup_pair_df), (pairs_after["iou"] >= DUP_IOU).sum()),
            ("맞닿은 쌍", len(touch_df), (pairs_after["iou"] < DUP_IOU).sum()),
            ("빈 배경 박스", len(empty_df), (diff_after < EMPTY_DIFF_MAX).sum()),
        ],
        columns=["검사", "[원본]", "[보정]"],
    ).set_index("검사")
    display(compare_df)

    counts = (
        pd.DataFrame(
            {
                "[원본]": ann_df["category_id"].value_counts(),
                "[보정]": ann_corrected_df["category_id"].value_counts(),
            }
        )
        .fillna(0)
        .astype(int)
    )
    counts = counts[counts["[원본]"] != counts["[보정]"]].join(
        cat_df.set_index("category_id")["dl_name"]
    )
    print("[보정] bbox 수가 바뀐 클래스")
    display(counts)

    remaining = ann_corrected_df[flags_after.any(axis=1)].join(flags_after)
    print("[보정] 남은 좌표·area 오류 (correction_ids가 비면 적용된 보정이 없는 라벨)")
    display(
        remaining[
            [
                "file_name",
                "annotation_id",
                "category_id",
                *BBOX_COLS,
                "area",
                "correction_ids",
                *flags_after.columns,
            ]
        ]
    )

In [ ]:
# 적용된 보정의 전후. 빨간 점선: 고치기 전, 초록: 고친 뒤(또는 추가한) 박스, 회색: 나머지 라벨
applied = apply_log_df[apply_log_df["status"] == "적용"]
records = correction_df.set_index("correction_id")
if applied.empty:
    print(f"[보정] 전후 그림: {correction_status}")
else:
    fig, axes = image_grid(len(applied), cols=4)
    for ax, rec in zip(axes, applied.itertuples(), strict=True):
        record = records.loc[rec.correction_id]
        boxes = ann_corrected_df[ann_corrected_df["file_name"] == rec.file_name]
        is_target = boxes["correction_ids"].str.contains(rec.correction_id, regex=False)
        angle = boxes["camera_la"].iloc[0]
        title = f"[보정] {rec.file_name.split('_')[0]} ({angle}°)\n{rec.case}: {rec.category_id}"
        show_image(ax, TRAIN_IMG_DIR / rec.file_name, title)
        draw_bboxes(ax, boxes[~is_target], color="gray")
        if isinstance(record["old_bbox"], tuple):
            draw_bbox(ax, record["old_bbox"], color="red", linestyle="--")
        draw_bbox(ax, record["new_bbox"], str(rec.category_id), color="lime")
    plt.tight_layout()
    plt.show()

아래는 kwon CSV가 있는 환경에서 관측한 값입니다(원본 상태 관측값과 보정 기록 기준).

- kwon 기록 13개 중 12개(추가 8개, 위치 수정 4개)가 적용되어 bbox는 763개 → 771개입니다. 라벨 누락 8개 → 0개,
  좌표 중복 3쌍 → 0쌍, 빈 배경 1개 → 0개입니다. 위치 수정은 bbox 수를 바꾸지 않으므로 늘어난 8개는 모두 라벨 누락 추가입니다.
- 이미지 밖 좌표 1개(18357)는 남습니다. kwon과 JJAM 방식이 같은 라벨에 다른 값(IoU 약 0.98)을 내서 충돌로 보류했기
  때문입니다. 값을 하나 고르고 다른 기록을 빼면 적용됩니다.
- area 불일치 6개도 남습니다. 19232 제안은 미검수이고, 나머지 5개는 보정 기록이 없습니다.
- 여백 보정 기록은 모두 미검수라 적용하지 않았습니다. 위치가 고쳐진 박스의 여백 기록은 현재 bbox와 `old_bbox`가 달라
  거부 이유가 함께 남습니다. 고친 좌표로 다시 계산해야 합니다(계획 6장).

**9장으로 넘길 결정**

1. 18357 bbox 값: kwon `[574, 626, 300, 309]`과 JJAM 색상 윤곽 제안 중 하나
2. 검수 기준: kwon 기록을 검수됨으로 볼지(`KWON_REVIEWED`), JJAM 19232 제안을 적용할지
3. 정답 박스의 여백 기준: 여백 20% 이상 박스만 줄일지, 그대로 둘지
4. 보정 기록 파일: `correction_df` 형식으로 `data/processed/`에 저장하는 작업을 `data_process/`에 만들지
   (저장하면 새 라벨 ID도 기록에 고정됩니다)

## 6. 분포

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 6.1 클래스별 bbox 수와 불균형 단계 — kwon 35, 37, jang 15
- 6.2 bbox 크기 분포, 클래스별 크기 — jang 13, 17

## 7. 촬영 조건

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 7.1 조합별 각도 구성 — kwon 41, 43
- 7.2 각도·배경·조명별 bbox 크기와 비율 — dongwook 11, jang 19–20
- 7.3 알약 실제 크기와 픽셀 크기 — dongwook 13–17 (JJAM 과거 분석은 Git 이력에서 확인 후 보완)

## 8. 데이터 누수와 분할

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 8.1 희귀 약은 사실상 장면 하나 — kwon 46–53
- 8.2 같은 조합 유사도 (pHash) — kwon 90–107, sohee_eda 15의 aHash와 비교
- 8.3 다른 조합의 세팅 공유와 묶음 — kwon 109–118
- 8.4 분할 후보 비교 — kwon 120, sohee_eda 17, jang 20

## 9. 결론과 다음 작업

> 이식 예정입니다. 아래 항목과 기준본 셀(원본 노트북 셀 번호)을 옮깁니다.

- 확정된 사실
- 결정할 것
- 학습 파이프라인으로 넘길 것